# Variant 4 - Curriculum GSM -> MATH t? baseline GPT-2 Math V7


## Run

Kaggle: GPU ON, Internet OFF. Tổng thời gian dự kiến **<= 3 giờ**.

Output:
- `data/train_preprocessed.json`: train set sau preprocessing và smart truncation.
- `data/train_preprocessing_report.json`: thống kê preprocessing.
- `gpt2_math_baseline_ckpt/`: checkpoint sau fine-tune.
- `valid_output.json` + `valid_report.json`: output và đánh giá chi tiết validation.
- `test_predictions.json`: file nộp cho Phase 2.


In [ ]:
# 1. Import và kiểm tra môi trường
import os
import sys
import gc
import re
import json
import math
import time
import random
import hashlib
import inspect
import platform
import shutil
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
    set_seed,
)

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from peft import LoraConfig, get_peft_model, PeftModel
    PEFT_AVAILABLE = True
except Exception as e:
    PEFT_AVAILABLE = False
    LoraConfig = None
    get_peft_model = None
    PeftModel = None
    print("WARNING: peft chưa khả dụng:", repr(e))
    
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 100)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Không bật deterministic mặc định vì có thể làm chậm training trên Kaggle.
# Nếu cần reproduce chặt hơn, có thể bật ở cell config:
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False

print("Python:", sys.version.replace("\n", " "))
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
CUDA_OK = torch.cuda.is_available()
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        capability = torch.cuda.get_device_capability(i)
        print(f"GPU {i}:", name, "| capability:", capability)
    major, minor = torch.cuda.get_device_capability(0)
    if major < 7:
        CUDA_OK = False
        print("WARNING: GPU hiện tại có compute capability < 7.0, không tương thích với PyTorch CUDA hiện tại.")
        print("Hãy chọn GPU T4/V100/A100 thay vì P100 trên Kaggle.")

In [ ]:
# 2. Đường dẫn dữ liệu, model và output
def first_existing(*paths):
    checked = []
    for p in paths:
        p = Path(p)
        checked.append(str(p))
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào:\n" + "\n".join(checked))


def first_existing_optional(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


IS_KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
    PROJECT_ROOT / "data",
    PROJECT_ROOT,
)

MODEL_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/nlphust-gpt2-vietnamese",
    PROJECT_ROOT / "models" / "nlphust-gpt2-vietnamese",
)

VARIANT_NAME = "variant_4_curriculum_gsm_to_math"
WORK_DIR = (Path("/kaggle/working") / VARIANT_NAME) if IS_KAGGLE else PROJECT_ROOT / "outputs" / VARIANT_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Keep generated artifacts isolated from baseline/other variants.
GENERATED_DATA_DIR = WORK_DIR / "data"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"
TEST_FILE = first_existing_optional(DATA_DIR / "test.json", "/kaggle/input/test.json")

STAGE_OUTPUT_ROOT = WORK_DIR / "checkpoints"
STAGE_PREDICTION_DIR = WORK_DIR / "stage_predictions"
STAGE_REPORT_DIR = WORK_DIR / "stage_reports"
for _p in [STAGE_OUTPUT_ROOT, STAGE_PREDICTION_DIR, STAGE_REPORT_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = STAGE_OUTPUT_ROOT / "stage4_all"
VALID_OUTPUT_PATH = STAGE_PREDICTION_DIR / "stage4_all_valid_output.json"
VALID_REPORT_PATH = WORK_DIR / "variant_4_curriculum_report.json"
CURRICULUM_REPORT_PATH = VALID_REPORT_PATH
TEST_OUTPUT_PATH = WORK_DIR / "test_predictions.json"

SAFE_EOS_ID = 50256
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("WORK_DIR:", WORK_DIR)
print("GENERATED_DATA_DIR:", GENERATED_DATA_DIR)
print("TEST_FILE:", TEST_FILE)


In [ ]:
# 3. Đọc dữ liệu
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
MAX_TEST_SAMPLES = None


def load_records(path, need_response=False):
    path = Path(path)
    with path.open("r", encoding="utf-8-sig") as f:
        first = f.read(1)
        f.seek(0)
        records = json.load(f) if first == "[" else [json.loads(line) for line in f if line.strip()]

    out = []
    for i, rec in enumerate(records):
        if "query_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu query_vi")
        if need_response and "response_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu response_vi")
        item = dict(rec)
        item.setdefault("id", i)
        item.setdefault("type", "unknown")
        out.append(item)
    return out


raw_train = load_records(TRAIN_FILE, need_response=True)
raw_valid = load_records(VALID_FILE, need_response=True) if VALID_FILE.exists() else []
raw_test = load_records(TEST_FILE) if TEST_FILE else []

if MAX_TRAIN_SAMPLES is not None:
    raw_train = raw_train[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES is not None:
    raw_valid = raw_valid[:MAX_VALID_SAMPLES]
if MAX_TEST_SAMPLES is not None:
    raw_test = raw_test[:MAX_TEST_SAMPLES]

print("raw train:", len(raw_train))
print("raw valid:", len(raw_valid))
print("raw test :", len(raw_test))
print(json.dumps(raw_train[0], ensure_ascii=False, indent=2)[:1400])

In [ ]:
# 4. Hàm trích đáp án và tính điểm
ANSWER_ANCHORS = [
    r"Đáp\s*án\s*là",
    r"Câu\s*trả\s*lời\s*là",
    r"(?:Câu\s+)?Trả\s*lời(?:\s+là)?",
    r"Đáp\s*án(?:\s+[A-Za-z]{1,4}\d{0,2})?",  # bắt cả "Đáp án C4:"
    r"Kết\s*quả\s*là",
    r"Vậy\s*đáp\s*án\s*là",
    r"The answer is",
    r"Answer",
    r"####",
]

ANSWER_ANCHOR_RE = re.compile(
    r"(?:"
    + "|".join(ANSWER_ANCHORS)
    + r")\s*[:：]?",
    flags=re.IGNORECASE,
)

BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
NUM_RE = re.compile(r"[-+]?\d[\d.,]*(?:\s*/\s*[-+]?\d[\d.,]*)?")


def clean_answer_tail(text):
    if text is None:
        return None
    text = str(text).strip().split("\n", 1)[0]
    text = re.sub(r"^(?:là|=|:|：)\s*", "", text, flags=re.IGNORECASE)
    text = text.strip(" .。;；,，]}）)")
    text = text.strip("[{(（")
    return text or None


def first_answer_unit(text):
    """
    Lấy đơn vị đáp án đầu tiên trong một đoạn tail sau anchor.
    Ưu tiên boxed, sau đó số đầu tiên.
    Không lấy số cuối toàn output nữa.
    """
    text = str(text or "")

    box = BOXED_RE.search(text)
    if box:
        return clean_answer_tail(box.group(1))

    num = NUM_RE.search(text)
    if num:
        return clean_answer_tail(num.group(0))

    return None


def last_answer_unit(text):
    """
    Fallback chỉ dùng khi không có anchor.
    """
    text = str(text or "")

    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])

    nums = NUM_RE.findall(text)
    if nums:
        return clean_answer_tail(nums[-1])

    return None


def extract_answer_text(text, allow_last_number=False, prefer_first_anchor=False):
    """
    - Với gold/reference: prefer_first_anchor=False để lấy anchor cuối nếu response có nhiều marker.
    - Với model output: prefer_first_anchor=True để lấy đáp án đầu tiên sau anchor, tránh đuôi rác kiểu
      'Đáp án: 9.5.5.5.8.8...'.
    """
    text = str(text or "")
    matches = list(ANSWER_ANCHOR_RE.finditer(text))

    if matches:
        m = matches[0] if prefer_first_anchor else matches[-1]
        tail = text[m.end():]
        ans = first_answer_unit(tail)
        if ans is not None:
            return ans

    if allow_last_number:
        return last_answer_unit(text)

    return None


def parse_plain_number(text):
    text = str(text).strip().replace(" ", "")
    if not text:
        return None

    if "/" in text:
        parts = text.split("/")
        if len(parts) == 2:
            a = parse_plain_number(parts[0])
            b = parse_plain_number(parts[1])
            if a is not None and b not in (None, 0):
                return a / b
        return None

    if re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", text):
        text = text.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", text):
        text = text.replace(",", "")
    elif "," in text and "." not in text:
        right = text.split(",")[-1]
        text = text.replace(",", "") if len(right) == 3 else text.replace(",", ".")
    elif "," in text and "." in text:
        text = text.replace(",", "")

    try:
        out = float(text)
    except ValueError:
        return None
    return out if math.isfinite(out) else None


def parse_number(text):
    if text is None:
        return None
    text = str(text).strip()
    if not text:
        return None
    direct = parse_plain_number(text)
    if direct is not None:
        return direct
    m = NUM_RE.search(text)
    return parse_plain_number(m.group(0)) if m else None


def relative_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))


def score_one(rel_err, extractable=True):
    if not extractable or rel_err is None:
        return 0
    if rel_err <= 0.01:
        return 10
    if rel_err <= 0.10:
        return 5
    if rel_err <= 0.50:
        return 1
    return 0

In [ ]:
# 5. Data processing trước khi train (Bước 1, 2, 3 + dedup nhẹ)
DROP_TRAIN_WITHOUT_FINAL_ANSWER = True
SAVE_PREPROCESSED_TRAIN = True
DEDUP_TRAIN = True   # V2: bật dedup nhẹ để giảm noise
PREPROCESSED_TRAIN_FILE = GENERATED_DATA_DIR / "train_preprocessed.json"
PREPROCESSING_REPORT_FILE = GENERATED_DATA_DIR / "train_preprocessing_report.json"

# Fast preset: reduce only large/easier types while keeping small/hard types intact.
# Large types are selected by TF-IDF similarity clusters, not by random sampling.
FAST_TRAIN_SUBSET = True
FAST_SUBSET_STRATEGY = "keep_hard_types_tfidf_cluster_diversity"
KEEP_ALL_TYPES_FOR_SPEED = {"GSM_FOBAR", "GSM_SV", "MATH_FOBAR", "MATH_SV"}
TYPE_QUOTAS_FOR_SPEED = {
    "GSM_AnsAug": 15000,
    "GSM_Rephrased": 15000,
    "MATH_AnsAug": 14000,
    "MATH_Rephrased": 10000,
}
MAX_TRAIN_RECORDS_FOR_SPEED = sum(TYPE_QUOTAS_FOR_SPEED.values())  # soft cap for large types only
SPEED_SAMPLE_SEED = SEED
DIVERSITY_MAX_FEATURES = 8192
DIVERSITY_MAX_CLUSTERS = 256
DIVERSITY_MIN_CLUSTERS = 32


def normalize_space(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def word_count(text):
    return len(re.findall(r"\S+", str(text or "")))


def normalized_hash(text):
    text = normalize_space(text).lower()
    return hashlib.blake2b(text.encode("utf-8"), digest_size=16).hexdigest()


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_records_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def load_jsonl_records(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def dataset_fingerprint(records):
    content = json.dumps(
        [r.get("query_vi", "") + "\n" + r.get("response_vi", "") for r in records],
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.md5(content).hexdigest()


def fix_artifacts(text):
    text = str(text or "")
    text = text.replace(r"\đóng hộp{", r"\boxed{")
    text = text.replace("\u200b", "").replace("\ufeff", "")
    return text.strip()


def strip_asy_blocks(text):
    text = str(text or "")
    text = re.sub(r"\[asy\].*?\[/asy\]", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(
        r"\[asy\].*?(?=(?:Giá trị của|Giá trị là|Câu trả lời|Đáp án|Nếu chúng ta biết|Để giải|$))",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    text = re.sub(r"\[/asy\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def clean_text(text):
    text = str(text or "")
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(Giá trị của biến [^\n?]+\?)\s*\1", r"\1", text)
    if re.search(r"Đáp án là|Câu trả lời là|####|\\boxed", text, flags=re.IGNORECASE):
        text = re.sub(
            r"\n(?:The answer is[:\s]+[\d.,/\\{}a-zA-Z]+\.?\s*)+$",
            "",
            text,
            flags=re.IGNORECASE,
        )
    return text.strip()


def normalize_decimal_format(text):
    text = str(text or "")
    text = re.sub(
        r"(?<![{\\])(\d+)\.(\d{3}),(\d{1,3})(?!\d)",
        lambda m: f"{m.group(1)}{m.group(2)}.{m.group(3)}",
        text,
    )
    text = re.sub(
        r"(?<![a-zA-ZÀ-ỹ{])(-?0),(\d{1,3})(?!\d)(?!})",
        lambda m: f"{m.group(1)}.{m.group(2)}",
        text,
    )
    text = re.sub(
        r"(?<![a-zA-ZÀ-ỹ{])(-?\d+),(\d{1,2})(?!\d)(?!})",
        lambda m: f"{m.group(1)}.{m.group(2)}",
        text,
    )
    return text


def preprocess_step2(query, response):
    query = strip_asy_blocks(query)
    response = strip_asy_blocks(response)
    query = clean_text(query)
    response = clean_text(response)
    query = normalize_decimal_format(query)
    response = normalize_decimal_format(response)
    return query, response


def extract_final_answer(response):
    text = str(response or "")
    anchor_re = re.compile(
        r"(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer|####)\s*[:：]?",
        flags=re.IGNORECASE,
    )
    matches = list(anchor_re.finditer(text))
    if matches:
        return clean_answer_tail(text[matches[-1].end():])

    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])

    numbers = re.findall(
        r"(?:\\frac\{[^}]+\}\{[^}]+\}|[-+]?\d+(?:[.,]\d+)?(?:\s*\\[a-zA-Z]+\{[^}]*\})*)",
        text,
    )
    if numbers:
        return clean_answer_tail(numbers[-1])
    return None


def normalize_answer(answer):
    answer = clean_answer_tail(answer) or ""
    answer = re.sub(r"\s+", " ", answer).strip()
    answer = re.sub(r"\(([-+]?\d+),([-+]?\d+)\)", r"(\1, \2)", answer)

    if not re.search(r"[\\{^_]", answer):
        if re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", answer) and not re.fullmatch(r"[-+]?0,\d{3}", answer):
            answer = answer.replace(",", "")
        elif re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", answer):
            answer = answer.replace(".", "").replace(",", ".")
        else:
            answer = normalize_decimal_format(answer)
        answer = re.sub(r"^([-+]?\d[\d./]*)(?:\s+[a-zA-ZÀ-ỹ%].*)$", r"\1", answer)
    else:
        answer = normalize_decimal_format(answer)

    return answer.strip(" .。;；,，")


def rebuild_response(response, answer):
    cleaned = str(response or "").strip()
    cleaned = re.sub(
        r"\s*(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer)\s*[:：]?\s*[^\n]*\s*$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\s*####\s*[^\n]*\s*$", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\n?\s*\\boxed\s*\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}\s*[.。]?\s*$", "", cleaned)
    cleaned = cleaned.rstrip()
    return (cleaned + f"\nĐáp án là: {answer}").strip()


def preprocess_labeled_record(rec, idx, drop_without_answer):
    query_raw = fix_artifacts(rec.get("query_vi"))
    response_raw = fix_artifacts(rec.get("response_vi"))
    if not query_raw or not response_raw:
        return None, "missing_query_or_response"

    query, response = preprocess_step2(query_raw, response_raw)
    answer = normalize_answer(extract_final_answer(response))
    if not answer and drop_without_answer:
        return None, "extract_failed"

    if answer:
        response = rebuild_response(response, answer)

    item = {
        "id": rec.get("id", idx),
        "query_vi": query,
        "response_vi": response,
        "type": rec.get("type", "unknown"),
        "answer_text": answer or None,
        "answer_num": parse_number(answer) if answer else None,
    }
    return item, None


def process_train(records):
    kept = []
    drop_reasons = []
    failed_samples = []
    asy_stripped = 0

    for i, rec in enumerate(tqdm(records, desc="text preprocessing")):
        raw_joined = f"{rec.get('query_vi', '')}\n{rec.get('response_vi', '')}".lower()
        had_asy = "[asy]" in raw_joined
        item, reason = preprocess_labeled_record(rec, i, DROP_TRAIN_WITHOUT_FINAL_ANSWER)
        if reason:
            drop_reasons.append(reason)
            if reason == "extract_failed" and len(failed_samples) < 50:
                failed_samples.append({
                    "index": i,
                    "type": rec.get("type", "unknown"),
                    "query_vi": normalize_space(rec.get("query_vi"))[:180],
                    "response_tail": str(rec.get("response_vi", ""))[-300:],
                })
            continue
        if had_asy:
            asy_stripped += 1
        kept.append(item)

    return kept, Counter(drop_reasons), {
        "extract_failed_preview": failed_samples,
        "asy_stripped_count": asy_stripped,
    }


DEDUP_KEY_MODE = "query_response"  # options: query_response, query_answer, original_response


def count_by_type(records):
    return dict(Counter(r.get("type", "unknown") for r in records))


def make_dedup_key(rec):
    q = normalize_space(rec.get("query_vi", "")).lower()
    answer = str(rec.get("answer_text", "")).strip()
    response = normalize_space(rec.get("response_vi", "")).lower()

    original_hash = rec.get("original_question_hash")
    response_hash = normalized_hash(response)

    if DEDUP_KEY_MODE == "query_response":
        # Khuyến nghị V3: giữ đa dạng lời giải, không drop mạnh các bài cùng đáp án.
        return (q, response)

    if DEDUP_KEY_MODE == "original_response" and original_hash:
        return (str(original_hash), response_hash)

    # fallback V2 cũ
    return (q, answer)


def dedup_records(records):
    """
    V3: dedup theo (query_norm, response_norm), không còn theo (query_norm, answer_text).
    Mục tiêu: tránh drop quá mạnh AnsAug chỉ vì cùng đáp án.
    """
    seen = set()
    out = []
    dup = 0
    dropped_by_type = Counter()

    before_by_type = Counter(r.get("type", "unknown") for r in records)

    for rec in records:
        key = make_dedup_key(rec)
        rec_type = rec.get("type", "unknown")

        if key in seen:
            dup += 1
            dropped_by_type[rec_type] += 1
            continue

        seen.add(key)
        out.append(rec)

    after_by_type = Counter(r.get("type", "unknown") for r in out)

    report = {
        "dedup_key_mode": DEDUP_KEY_MODE,
        "before_by_type": dict(before_by_type),
        "after_by_type": dict(after_by_type),
        "dropped_by_type": dict(dropped_by_type),
        "kept_ratio_by_type": {
            t: round(after_by_type.get(t, 0) / max(1, before_by_type.get(t, 0)), 4)
            for t in sorted(before_by_type)
        },
    }

    return out, dup, report



def _allocate_cluster_quotas(labels, target, total_count):
    cluster_counts = Counter(labels)
    raw_alloc = {
        label: cluster_counts[label] * target / max(1, total_count)
        for label in cluster_counts
    }
    alloc = {
        label: min(cluster_counts[label], int(math.floor(raw_alloc[label])))
        for label in cluster_counts
    }

    if target >= len(cluster_counts):
        for label in cluster_counts:
            if alloc[label] == 0:
                alloc[label] = 1

    current = sum(alloc.values())
    while current < target:
        candidates = [
            label for label in cluster_counts
            if alloc[label] < cluster_counts[label]
        ]
        if not candidates:
            break
        label = max(
            candidates,
            key=lambda x: (raw_alloc[x] - math.floor(raw_alloc[x]), cluster_counts[x] - alloc[x]),
        )
        alloc[label] += 1
        current += 1

    while current > target:
        candidates = [label for label in cluster_counts if alloc[label] > 0]
        if not candidates:
            break
        label = min(candidates, key=lambda x: (raw_alloc[x] - math.floor(raw_alloc[x]), alloc[x]))
        alloc[label] -= 1
        current -= 1

    return alloc


def _fallback_diverse_select(items, target):
    """Deterministic fallback when sklearn is unavailable."""
    buckets = defaultdict(list)
    for rec in items:
        q_words = word_count(rec.get("query_vi", ""))
        r_words = word_count(rec.get("response_vi", ""))
        answer_num = parse_number(rec.get("answer_text"))
        answer_bucket = "none" if answer_num is None else int(min(20, abs(answer_num) // 10))
        key = (q_words // 20, r_words // 40, answer_bucket)
        buckets[key].append(rec)

    for key in buckets:
        buckets[key].sort(key=lambda r: normalized_hash(r.get("query_vi", "") + "\n" + r.get("response_vi", "")))

    selected = []
    keys = sorted(buckets)
    while len(selected) < target and keys:
        next_keys = []
        for key in keys:
            if buckets[key] and len(selected) < target:
                selected.append(buckets[key].pop(0))
            if buckets[key]:
                next_keys.append(key)
        keys = next_keys
    return selected


def select_diverse_by_tfidf_clusters(items, target, rec_type, seed=SEED):
    """Select representative samples using TF-IDF similarity clusters."""
    items = list(items)
    if target >= len(items):
        return items, {
            "selector": "keep_all",
            "before": len(items),
            "after": len(items),
            "target": target,
        }
    if target <= 0:
        return [], {
            "selector": "drop_all",
            "before": len(items),
            "after": 0,
            "target": target,
        }

    try:
        from sklearn.cluster import MiniBatchKMeans
        from sklearn.feature_extraction.text import TfidfVectorizer

        texts = [normalize_space(r.get("query_vi", "")) for r in items]
        vectorizer = TfidfVectorizer(
            max_features=DIVERSITY_MAX_FEATURES,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            norm="l2",
        )
        x = vectorizer.fit_transform(texts)
        n_clusters = min(
            len(items),
            target,
            DIVERSITY_MAX_CLUSTERS,
            max(DIVERSITY_MIN_CLUSTERS, target // 60),
        )
        n_clusters = max(1, int(n_clusters))

        kmeans = MiniBatchKMeans(
            n_clusters=n_clusters,
            random_state=seed,
            batch_size=2048,
            n_init=3,
            max_iter=80,
        )
        labels = kmeans.fit_predict(x)
        distances = kmeans.transform(x)[np.arange(len(items)), labels]
        alloc = _allocate_cluster_quotas(labels, target, len(items))

        by_cluster = defaultdict(list)
        for i, label in enumerate(labels):
            rec = items[i]
            tie = normalized_hash(rec.get("query_vi", "") + "\n" + rec.get("response_vi", ""))
            by_cluster[int(label)].append((float(distances[i]), tie, rec))

        selected = []
        for label in sorted(by_cluster):
            rows = sorted(by_cluster[label], key=lambda x: (x[0], x[1]))
            selected.extend([rec for _, _, rec in rows[:alloc.get(label, 0)]])

        if len(selected) < target:
            selected_ids = {id(rec) for rec in selected}
            leftovers = [
                (normalized_hash(r.get("query_vi", "") + "\n" + r.get("response_vi", "")), r)
                for r in items
                if id(r) not in selected_ids
            ]
            leftovers.sort(key=lambda x: x[0])
            selected.extend([r for _, r in leftovers[: target - len(selected)]])

        return selected[:target], {
            "selector": "tfidf_minibatch_kmeans",
            "before": len(items),
            "after": min(target, len(selected)),
            "target": target,
            "n_clusters": n_clusters,
            "max_features": DIVERSITY_MAX_FEATURES,
        }
    except Exception as e:
        selected = _fallback_diverse_select(items, target)
        return selected, {
            "selector": "fallback_length_answer_buckets",
            "before": len(items),
            "after": len(selected),
            "target": target,
            "error": repr(e),
        }


def diversity_limit_records(records, keep_all_types, type_quotas, seed=SEED, type_key="type"):
    records = list(records)
    groups = defaultdict(list)
    for rec in records:
        groups[str(rec.get(type_key, "unknown"))].append(rec)

    selected = []
    selection_by_type = {}
    before_by_type = {t: len(v) for t, v in groups.items()}

    for rec_type in sorted(groups):
        items = groups[rec_type]
        if rec_type in keep_all_types:
            chosen = list(items)
            info = {
                "selector": "keep_all_hard_or_small_type",
                "before": len(items),
                "after": len(chosen),
                "target": len(items),
            }
        else:
            target = min(len(items), int(type_quotas.get(rec_type, len(items))))
            chosen, info = select_diverse_by_tfidf_clusters(items, target, rec_type, seed=seed)
        selected.extend(chosen)
        selection_by_type[rec_type] = info

    selected.sort(key=lambda r: (str(r.get("type", "unknown")), normalized_hash(r.get("query_vi", "") + "\n" + r.get("response_vi", ""))))
    after_by_type = count_by_type(selected)
    report = {
        "enabled": True,
        "strategy": FAST_SUBSET_STRATEGY,
        "before": len(records),
        "after": len(selected),
        "keep_all_types": sorted(keep_all_types),
        "type_quotas": dict(type_quotas),
        "before_by_type": before_by_type,
        "after_by_type": after_by_type,
        "kept_ratio_by_type": {
            t: round(after_by_type.get(t, 0) / max(1, before_by_type.get(t, 0)), 4)
            for t in sorted(before_by_type)
        },
        "selection_by_type": selection_by_type,
    }
    return selected, report


def process_eval_or_test(records, has_response):
    out = []
    for i, rec in enumerate(records):
        query_raw = fix_artifacts(rec.get("query_vi"))
        query = normalize_decimal_format(clean_text(strip_asy_blocks(query_raw)))
        item = {
            "id": rec.get("id", i),
            "query_vi": query,
            "type": rec.get("type", "unknown"),
        }
        if has_response:
            response_raw = fix_artifacts(rec.get("response_vi"))
            _, response = preprocess_step2(query_raw, response_raw)
            answer = normalize_answer(extract_final_answer(response))
            if answer:
                response = rebuild_response(response, answer)
            item["response_vi"] = response
            item["answer_text"] = answer or None
            item["answer_num"] = parse_number(answer) if answer else None
        out.append(item)
    return out


train_records, drop_counter, preprocess_logs = process_train(raw_train)
n_before_dedup = len(train_records)

if DEDUP_TRAIN:
    train_records, n_dup, dedup_report = dedup_records(train_records)
    print(f"Dedup mode={DEDUP_KEY_MODE}: bỏ {n_dup} mẫu trùng. Còn {len(train_records)} mẫu.")
    print("Dedup kept_ratio_by_type:")
    print(json.dumps(dedup_report["kept_ratio_by_type"], ensure_ascii=False, indent=2))
else:
    n_dup = 0
    dedup_report = {
        "dedup_key_mode": None,
        "before_by_type": count_by_type(train_records),
        "after_by_type": count_by_type(train_records),
        "dropped_by_type": {},
        "kept_ratio_by_type": {},
    }

if FAST_TRAIN_SUBSET:
    train_records, speed_subset_report = diversity_limit_records(
        train_records,
        KEEP_ALL_TYPES_FOR_SPEED,
        TYPE_QUOTAS_FOR_SPEED,
        seed=SPEED_SAMPLE_SEED,
    )
    print("Fast diversity subset:", speed_subset_report["before"], "->", speed_subset_report["after"])
    print("Fast subset kept_ratio_by_type:")
    print(json.dumps(speed_subset_report["kept_ratio_by_type"], ensure_ascii=False, indent=2))
    print("Fast subset selectors by type:")
    print(json.dumps(speed_subset_report["selection_by_type"], ensure_ascii=False, indent=2)[:4000])
else:
    speed_subset_report = {
        "enabled": False,
        "strategy": "none",
        "before": len(train_records),
        "after": len(train_records),
        "keep_all_types": [],
        "type_quotas": {},
        "before_by_type": count_by_type(train_records),
        "after_by_type": count_by_type(train_records),
    }
    

valid_records = process_eval_or_test(raw_valid, has_response=True)
test_records = process_eval_or_test(raw_test, has_response=False)

preprocessing_report = {
    "train_before": len(raw_train),
    "train_after_text_preprocessing": n_before_dedup,
    "train_after_dedup": speed_subset_report.get("before", len(train_records)),
    "train_after_speed_subset": len(train_records),
    "speed_subset_report": speed_subset_report,
    "dropped_text_preprocessing": sum(drop_counter.values()),
    "drop_reasons_text_preprocessing": dict(drop_counter),
    "n_duplicates_removed": n_dup,
    "valid": len(valid_records),
    "test": len(test_records),
    "fingerprint_text_preprocessing": dataset_fingerprint(train_records),
    "dedup_report": dedup_report,
    "dedup_key_mode": DEDUP_KEY_MODE,
    **preprocess_logs,
}

print("train before:", len(raw_train), "| after text preprocessing:", n_before_dedup, "| dropped:", sum(drop_counter.values()))
print("train after dedup/subset:", speed_subset_report.get("before"), "->", len(train_records))
print("drop reasons:", dict(drop_counter))
print("[asy] stripped in train:", preprocess_logs["asy_stripped_count"])
print("valid:", len(valid_records), "| test:", len(test_records))
print("File train mới sẽ được ghi sau smart truncation:", PREPROCESSED_TRAIN_FILE)
print("\nTarget sau xử lý:")
print(train_records[0]["response_vi"][:800])




In [ ]:
# 6. Kiểm tra dữ liệu sau processing
def feature_df(records, split):
    rows = []
    for i, rec in enumerate(records):
        rows.append({
            "split": split,
            "index": i,
            "type": rec.get("type", "unknown"),
            "query_words": word_count(rec.get("query_vi")),
            "response_words": word_count(rec.get("response_vi", "")),
            "has_answer": rec.get("answer_text") is not None,
            "answer_num": rec.get("answer_num"),
            "ends_with_anchor": str(rec.get("response_vi", "")).rstrip().endswith("Đáp án là: " + str(rec.get("answer_text", ""))),
        })
    return pd.DataFrame(rows)


train_df = feature_df(train_records, "train")
valid_df = feature_df(valid_records, "valid") if valid_records else pd.DataFrame()

print("Phân bố type sau processing:")
display(train_df["type"].value_counts().rename_axis("type").reset_index(name="count"))

print("Độ dài train (words):")
display(train_df[["query_words", "response_words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))

print("Độ dài và answer rate theo type:")
by_type = (
    train_df.groupby("type")
    .agg(
        count=("index", "count"),
        query_p95=("query_words", lambda s: s.quantile(0.95)),
        response_p95=("response_words", lambda s: s.quantile(0.95)),
        answer_rate=("has_answer", "mean"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)
display(by_type.round(3))

print("Tỷ lệ có final_answer:", round(float(train_df["has_answer"].mean()), 4))
print("Anchor không nằm cuối response:", int((~train_df["ends_with_anchor"]).sum()) if len(train_df) else 0)

In [ ]:
# 7. Prompt V2, tokenizer, smart truncation, token audit
MAX_LENGTH = 512
HARD_TRUNC_TYPES = {"MATH_SV", "MATH_FOBAR"}
TRUNCATION_MARKER = "\n...\n"
TOKEN_AUDIT_SAMPLES = 2000

# V3 PROMPT: explicit task-conditioning theo cấu trúc target
TASK_GROUP_MAP = {
    "GSM_AnsAug": "DIRECT_ANSWER",
    "GSM_Rephrased": "DIRECT_ANSWER",
    "MATH_AnsAug": "DIRECT_ANSWER",
    "MATH_Rephrased": "DIRECT_ANSWER",
    "GSM_SV": "SOLVE_FOR_VARIABLE",
    "MATH_SV": "SOLVE_FOR_VARIABLE",
    "GSM_FOBAR": "REVERSE_PARAM",
    "MATH_FOBAR": "REVERSE_PARAM",
    "unknown": "DIRECT_ANSWER",
}

TYPE_LABEL_MAP = {
    "GSM_AnsAug":     "Bài toán số học đời sống - hỏi đáp án trực tiếp",
    "GSM_Rephrased":  "Bài toán số học đời sống - hỏi đáp án trực tiếp",
    "MATH_AnsAug":    "Bài toán nâng cao - hỏi đáp án trực tiếp",
    "MATH_Rephrased": "Bài toán nâng cao - hỏi đáp án trực tiếp",
    "GSM_SV":         "Bài toán số học đời sống - tìm biến chưa biết",
    "MATH_SV":        "Bài toán nâng cao - tìm biến chưa biết",
    "GSM_FOBAR":      "Bài toán số học đời sống - suy ngược tham số",
    "MATH_FOBAR":     "Bài toán nâng cao - suy ngược tham số",
    "unknown":        "Bài toán",
}

TASK_INSTRUCTION_MAP = {
    "DIRECT_ANSWER": "Tìm đáp án cuối cùng đúng với câu hỏi trong đề.",
    "SOLVE_FOR_VARIABLE": "Tìm giá trị của biến/chưa biết được hỏi, không trả lời lại đáp án gốc nếu đề đã biến đổi mục tiêu.",
    "REVERSE_PARAM": "Suy ngược tham số hoặc dữ kiện cần thiếu sao cho điều kiện trong đề đúng.",
}

VALID_TARGET_MODES = {"solution", "answer_only", "mixed"}
TARGET_MODE = "solution"      # "solution" keeps v7 reasoning targets; "answer_only" trains direct answers; "mixed" blends both.
ANSWER_ONLY_RATIO = 1.0       # Used only when TARGET_MODE="mixed".

if TARGET_MODE not in VALID_TARGET_MODES:
    raise ValueError(f"TARGET_MODE must be one of {sorted(VALID_TARGET_MODES)}, got: {TARGET_MODE!r}")
ANSWER_ONLY_RATIO = float(ANSWER_ONLY_RATIO)
if not 0.0 <= ANSWER_ONLY_RATIO <= 1.0:
    raise ValueError(f"ANSWER_ONLY_RATIO must be in [0, 1], got: {ANSWER_ONLY_RATIO}")

SOLUTION_INSTRUCTION = 'Giải bài toán sau từng bước. Kết thúc bằng dòng "Đáp án là: <số>".'
ANSWER_ONLY_INSTRUCTION = 'Trả lời bài toán sau bằng đúng một dòng "Đáp án là: <số>". Không trình bày lời giải.'
MIXED_INSTRUCTION = 'Tìm đáp án cuối cùng của bài toán. Luôn kết thúc bằng dòng "Đáp án là: <số>".'
INSTRUCTION_BY_TARGET_MODE = {
    "solution": SOLUTION_INSTRUCTION,
    "answer_only": ANSWER_ONLY_INSTRUCTION,
    "mixed": MIXED_INSTRUCTION,
}
INSTRUCTION = INSTRUCTION_BY_TARGET_MODE[TARGET_MODE]

PROMPT_TEMPLATE = (
    "{instr}\n\n"
    "[TASK:{task_group}]\n"
    "[Loại: {type_label}]\n"
    "Mục tiêu: {task_instruction}\n"
    "Bài toán: {q}\n\n"
    "{response_prefix}"
)


def get_task_group(t):
    return TASK_GROUP_MAP.get(t, TASK_GROUP_MAP["unknown"])


def get_type_label(t):
    return TYPE_LABEL_MAP.get(t, TYPE_LABEL_MAP["unknown"])


def get_task_instruction(t):
    return TASK_INSTRUCTION_MAP[get_task_group(t)]


def _stable_unit_interval(*parts):
    key = "||".join(str(p) for p in parts)
    digest = hashlib.blake2b(key.encode("utf-8"), digest_size=8).hexdigest()
    return int(digest, 16) / float(16 ** 16)


def target_mode_for_record(rec):
    if TARGET_MODE == "solution":
        return "solution"
    if TARGET_MODE == "answer_only":
        return "answer_only"

    score = _stable_unit_interval(
        rec.get("id", ""),
        rec.get("type", "unknown"),
        rec.get("query_vi", ""),
        SEED,
    )
    return "answer_only" if score < ANSWER_ONLY_RATIO else "solution"


def get_prompt_instruction(rec):
    if TARGET_MODE != "mixed":
        return INSTRUCTION
    mode = target_mode_for_record(rec)
    return INSTRUCTION_BY_TARGET_MODE[mode]


def get_prompt_response_prefix(rec):
    return "Trả lời:\n" if target_mode_for_record(rec) == "answer_only" else "Lời giải:\n"


def build_answer_only_response(rec):
    answer = str(rec.get("answer_text") or "").strip()
    if not answer:
        answer = extract_answer_text(
            rec.get("response_vi", ""),
            allow_last_number=True,
            prefer_first_anchor=False,
        )
    if answer:
        return f"Đáp án là: {answer}"
    return str(rec.get("response_vi", "")).strip()


def build_target_response(rec):
    if target_mode_for_record(rec) == "answer_only":
        return build_answer_only_response(rec)
    return str(rec.get("response_vi", "")).strip()


def build_prompt(rec):
    rec_type = rec.get("type", "unknown")
    return PROMPT_TEMPLATE.format(
        instr=get_prompt_instruction(rec),
        task_group=get_task_group(rec_type),
        type_label=get_type_label(rec_type),
        task_instruction=get_task_instruction(rec_type),
        q=str(rec.get("query_vi", "")).strip(),
        response_prefix=get_prompt_response_prefix(rec),
    )


tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)

# Verify EOS thật của tokenizer; fallback về SAFE_EOS_ID nếu None.
# Lưu ý: checkpoint GPT-2 có thể có embedding nhỏ hơn len(tokenizer),
# nên model sẽ được resize ở cell train/inference trước khi dùng PAD/EOS này.
real_eos = tokenizer.eos_token_id
EOS_ID = int(real_eos) if real_eos is not None else SAFE_EOS_ID
PAD_ID = EOS_ID
tokenizer.pad_token_id = PAD_ID
tokenizer.eos_token_id = EOS_ID
if getattr(tokenizer, "pad_token", None) is None and getattr(tokenizer, "eos_token", None) is not None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocab_size:", getattr(tokenizer, "vocab_size", None), "| len:", len(tokenizer))
print(f"EOS_ID (used): {EOS_ID} | PAD_ID: {PAD_ID} | tokenizer.eos_token_id thật: {real_eos}")


def encode_no_special(text):
    return tokenizer(str(text or ""), add_special_tokens=False)["input_ids"]


def split_response_for_truncation(response, answer):
    response = str(response or "").rstrip()
    m = re.search(r"\nĐáp án là:\s*([^\n]+)\s*$", response, flags=re.IGNORECASE)
    if m:
        return response[:m.start()].rstrip(), "\nĐáp án là: " + m.group(1).strip()
    suffix = "\nĐáp án là: " + str(answer or extract_answer_text(response, allow_last_number=True) or "").strip()
    body = re.sub(r"\n?Đáp án là:\s*[^\n]+\s*$", "", response, flags=re.IGNORECASE).rstrip()
    return body, suffix


def measure_record_tokens(rec):
    prompt_ids = encode_no_special(build_prompt(rec))
    response_ids = encode_no_special(build_target_response(rec)) + [EOS_ID]
    return len(prompt_ids), len(response_ids), len(prompt_ids) + len(response_ids)


def decode_ids(ids):
    return tokenizer.decode(ids, skip_special_tokens=True).strip()


def make_truncated_body(body_ids, middle_budget, rec_type):
    """
    V3 truncation:
    - Không chỉ giữ tail như V2.
    - Với MATH_SV/MATH_FOBAR: giữ nhiều phần đầu hơn vì phần đầu thường chứa setup phương trình.
    - Vẫn giữ tail để không mất bước kết luận trước anchor.
    """
    if len(body_ids) <= middle_budget:
        return decode_ids(body_ids)

    if middle_budget <= 32:
        # Quá ít budget: fallback giữ tail như cũ.
        return decode_ids(body_ids[-middle_budget:])

    marker_ids = encode_no_special(TRUNCATION_MARKER)
    marker_budget = len(marker_ids)

    available = max(1, middle_budget - marker_budget)

    if rec_type in HARD_TRUNC_TYPES:
        head_budget = int(available * 0.70)
        tail_budget = available - head_budget
    else:
        head_budget = int(available * 0.45)
        tail_budget = available - head_budget

    head_budget = max(1, head_budget)
    tail_budget = max(1, tail_budget)

    head_text = decode_ids(body_ids[:head_budget])
    tail_text = decode_ids(body_ids[-tail_budget:])

    return (head_text + TRUNCATION_MARKER + tail_text).strip()


def smart_truncate_record(rec, max_length):
    target_mode = target_mode_for_record(rec)
    target_response = build_target_response(rec)
    prompt_ids = encode_no_special(build_prompt(rec))
    response_ids = encode_no_special(target_response) + [EOS_ID]
    original_length = len(prompt_ids) + len(response_ids)

    item = dict(rec)
    item["target_mode"] = target_mode
    item["response_vi"] = target_response
    item["original_length"] = original_length
    item["was_truncated"] = False
    item["truncation_strategy"] = "none"

    if original_length <= max_length:
        item["prompt_tokens"] = len(prompt_ids)
        item["response_tokens"] = len(response_ids)
        item["total_tokens"] = original_length
        return item, None

    body, suffix = split_response_for_truncation(target_response, rec.get("answer_text"))
    suffix_ids = encode_no_special(suffix) + [EOS_ID]
    middle_budget = max_length - len(prompt_ids) - len(suffix_ids)

    if middle_budget <= 0:
        return None, "too_long_prompt_or_answer"

    body_ids = encode_no_special(body)
    rec_type = rec.get("type", "unknown")

    while True:
        body_text = make_truncated_body(body_ids, middle_budget, rec_type)
        new_response = (body_text.rstrip() + suffix) if body_text else suffix.lstrip()
        new_response_ids = encode_no_special(new_response) + [EOS_ID]
        new_total = len(prompt_ids) + len(new_response_ids)

        if new_total <= max_length:
            item["response_vi"] = new_response
            item["was_truncated"] = True
            item["truncation_strategy"] = "head_tail_keep_answer_suffix"
            item["prompt_tokens"] = len(prompt_ids)
            item["response_tokens"] = len(new_response_ids)
            item["total_tokens"] = new_total
            return item, None

        middle_budget -= max(8, new_total - max_length)
        if middle_budget <= 0:
            return None, "too_long_after_truncation"


def apply_token_length_policy(records, max_length):
    kept = []
    counter = Counter()
    examples = []
    for rec in tqdm(records, desc="smart truncation"):
        item, reason = smart_truncate_record(rec, max_length)
        if reason:
            counter[reason] += 1
            if len(examples) < 20:
                examples.append({
                    "id": rec.get("id"),
                    "type": rec.get("type"),
                    "reason": reason,
                    "query_vi": rec.get("query_vi", "")[:180],
                })
            continue
        if item.get("was_truncated"):
            counter["smart_truncated"] += 1
        kept.append(item)
    return kept, counter, examples


train_records, token_policy_counter, token_policy_examples = apply_token_length_policy(train_records, MAX_LENGTH)
target_mode_counts_after_policy = Counter(target_mode_for_record(r) for r in train_records)
preprocessing_report.update({
    "max_length": MAX_LENGTH,
    "target_mode": TARGET_MODE,
    "answer_only_ratio": ANSWER_ONLY_RATIO,
    "target_mode_counts_after_policy": dict(target_mode_counts_after_policy),
    "train_after_token_policy": len(train_records),
    "dropped_token_policy": int(token_policy_counter.get("too_long_prompt_or_answer", 0) + token_policy_counter.get("too_long_after_truncation", 0)),
    "smart_truncated": int(token_policy_counter.get("smart_truncated", 0)),
    "token_policy_counter": dict(token_policy_counter),
    "token_policy_drop_preview": token_policy_examples,
    "fingerprint_final": dataset_fingerprint(train_records),
    "prompt_template_v3": PROMPT_TEMPLATE,
    "instruction": INSTRUCTION,
    "instruction_by_target_mode": INSTRUCTION_BY_TARGET_MODE,
    "type_label_map": TYPE_LABEL_MAP,
    "task_group_map": TASK_GROUP_MAP,
    "task_instruction_map": TASK_INSTRUCTION_MAP,
})

if SAVE_PREPROCESSED_TRAIN:
    save_records_jsonl(train_records, PREPROCESSED_TRAIN_FILE)
    save_json(preprocessing_report, PREPROCESSING_REPORT_FILE)
    print("Wrote:", PREPROCESSED_TRAIN_FILE)
    print("Wrote:", PREPROCESSING_REPORT_FILE)

    train_records = load_jsonl_records(PREPROCESSED_TRAIN_FILE)
    TRAIN_SOURCE = f"preprocessed_file:{PREPROCESSED_TRAIN_FILE}"
else:
    TRAIN_SOURCE = "preprocessed_in_memory"

print("TRAIN_SOURCE:", TRAIN_SOURCE)
print("Train records used by Trainer:", len(train_records))
assert train_records, "Không có mẫu train sau preprocessing"
assert max(r.get("total_tokens", 0) for r in train_records) <= MAX_LENGTH, "Còn mẫu vượt MAX_LENGTH"

sample = train_records if len(train_records) <= TOKEN_AUDIT_SAMPLES else random.sample(train_records, TOKEN_AUDIT_SAMPLES)
token_rows = []
for rec in tqdm(sample, desc="token audit"):
    p_tokens, r_tokens, total_tokens = measure_record_tokens(rec)
    token_rows.append({
        "type": rec.get("type"),
        "prompt_tokens": p_tokens,
        "response_tokens": r_tokens,
        "total_tokens": total_tokens,
        "will_truncate": total_tokens > MAX_LENGTH,
        "was_truncated": bool(rec.get("was_truncated")),
    })

token_df = pd.DataFrame(token_rows)

token_by_type = (
    token_df.groupby("type")
    .agg(
        n=("type", "size"),
        avg_total_tokens=("total_tokens", "mean"),
        p95_total_tokens=("total_tokens", lambda x: float(x.quantile(0.95))),
        truncated_rate=("was_truncated", "mean"),
    )
    .reset_index()
)

display(token_by_type.sort_values("truncated_rate", ascending=False))

preprocessing_report["token_policy_by_type"] = token_by_type.to_dict("records")
preprocessing_report["hard_trunc_types"] = sorted(HARD_TRUNC_TYPES)
preprocessing_report["truncation_marker"] = TRUNCATION_MARKER
save_json(preprocessing_report, PREPROCESSING_REPORT_FILE)
print("Updated preprocessing report with token_policy_by_type:", PREPROCESSING_REPORT_FILE)

display(token_df[["prompt_tokens", "response_tokens", "total_tokens"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))
print("Tỷ lệ còn vượt MAX_LENGTH:", round(float(token_df["will_truncate"].mean()), 4))
print("Số mẫu smart truncated:", int(token_policy_counter.get("smart_truncated", 0)))
print("Số mẫu drop vì quá dài:", preprocessing_report["dropped_token_policy"])

print("\n--- Sample prompt V2 ---")
print(build_prompt(train_records[0]))
print("--- end ---")


In [ ]:
# 8. Dataset cho supervised fine-tuning (loss mask trên prompt)
def clamp_ids(ids, vocab_size):
    return [min(max(int(x), 0), vocab_size - 1) for x in ids]


def fit_prompt_response(prompt_ids, response_ids, max_length):
    if len(prompt_ids) + len(response_ids) <= max_length:
        return prompt_ids, response_ids
    room = max_length - len(prompt_ids)
    if room <= 0:
        prompt_ids = prompt_ids[: max_length - 1]
        room = max_length - len(prompt_ids)
    response_ids = response_ids[-room:] if room > 0 else []
    return prompt_ids, response_ids


class MathDataset(Dataset):
    def __init__(self, records, tokenizer, vocab_size, max_length):
        self.records = records
        self.tokenizer = tokenizer
        self.vocab_size = vocab_size
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        prompt_ids = self.tokenizer(build_prompt(rec), add_special_tokens=False)["input_ids"]
        response_text = build_target_response(rec)
        response_ids = self.tokenizer(response_text, add_special_tokens=False)["input_ids"] + [EOS_ID]
        prompt_ids, response_ids = fit_prompt_response(prompt_ids, response_ids, self.max_length)

        input_ids = clamp_ids(prompt_ids + response_ids, self.vocab_size)
        labels = [-100] * len(prompt_ids) + clamp_ids(response_ids, self.vocab_size)
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }


@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        max_len = int(math.ceil(max_len / 8) * 8)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for item in batch:
            pad = max_len - len(item["input_ids"])
            out["input_ids"].append(item["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(item["attention_mask"] + [0] * pad)
            out["labels"].append(item["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


In [ ]:
# 9. Tham số train Variant 4 - curriculum GSM -> MATH theo type
RUN_TRAIN = True
EXPERIMENT_NAME = VARIANT_NAME

# Base training hyperparameters inherited from v7; curriculum is the only training-process change.
EPOCHS = 1                       # fallback/default per stage if stage-specific value is missing
PER_DEVICE_BATCH_SIZE = 8        # nếu OOM thì hạ lại 4
GRAD_ACCUM = 4                   # effective batch = 64 nếu có 2 GPU, =32 nếu 1 GPU
LEARNING_RATE = 7e-5
WARMUP_RATIO = 0.02
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 100

TRAINER_EVAL_SAMPLES = 256
EVAL_DURING_TRAIN = False        # full validation vẫn chạy sau mỗi stage checkpoint
EVAL_LOSS_AFTER_EACH_STAGE = True
SAVE_DURING_TRAIN = False        # save mỗi stage sau khi stage train xong

# Có thể chỉnh để kiểm soát thời gian chạy. max_steps=None nghĩa là chạy theo epochs.
CURRICULUM_STAGE_EPOCHS = {
    "stage1_GSM_direct": 1.0,
    "stage2_GSM_all": 1.0,
    "stage3_GSM_MATH_direct": 1.0,
    "stage4_all": 1.0,
}
CURRICULUM_STAGE_MAX_STEPS = {
    "stage1_GSM_direct": None,
    "stage2_GSM_all": None,
    "stage3_GSM_MATH_direct": None,
    "stage4_all": None,
}
CURRICULUM_STAGE_MAX_SAMPLES = {
    "stage1_GSM_direct": None,
    "stage2_GSM_all": None,
    "stage3_GSM_MATH_direct": None,
    "stage4_all": None,
}
CURRICULUM_SAMPLE_WITH_REPLACEMENT = False
CURRICULUM_RATIO_TOLERANCE = 1e-9
BASELINE_REFERENCE_SCORE_10 = None  # Có thể set 7.48 nếu muốn tính delta trực tiếp từ baseline 0.748.
BASELINE_REFERENCE_REPORT_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "baseline_gpt2_math_v2" / "valid_report.json",
]

CURRICULUM_STAGE_SPECS = [
    {
        "name": "stage1_GSM_direct",
        "types": ["GSM_AnsAug", "GSM_Rephrased"],
        "ratios": {"GSM_AnsAug": 0.50, "GSM_Rephrased": 0.50},
    },
    {
        "name": "stage2_GSM_all",
        "types": ["GSM_AnsAug", "GSM_Rephrased", "GSM_SV", "GSM_FOBAR"],
        "ratios": {"GSM_AnsAug": 0.35, "GSM_Rephrased": 0.35, "GSM_SV": 0.15, "GSM_FOBAR": 0.15},
    },
    {
        "name": "stage3_GSM_MATH_direct",
        "types": ["GSM_AnsAug", "GSM_Rephrased", "GSM_SV", "GSM_FOBAR", "MATH_AnsAug", "MATH_Rephrased"],
        "ratios": {
            "GSM_AnsAug": 0.25,
            "GSM_Rephrased": 0.25,
            "GSM_SV": 0.10,
            "GSM_FOBAR": 0.10,
            "MATH_AnsAug": 0.15,
            "MATH_Rephrased": 0.15,
        },
    },
    {
        "name": "stage4_all",
        "types": ["GSM_AnsAug", "GSM_Rephrased", "GSM_SV", "GSM_FOBAR", "MATH_AnsAug", "MATH_Rephrased", "MATH_SV", "MATH_FOBAR"],
        "ratios": {
            "GSM_AnsAug": 0.20,
            "GSM_Rephrased": 0.20,
            "GSM_SV": 0.10,
            "GSM_FOBAR": 0.10,
            "MATH_AnsAug": 0.15,
            "MATH_Rephrased": 0.15,
            "MATH_SV": 0.05,
            "MATH_FOBAR": 0.05,
        },
    },
]

# V3 LoRA capacity experiment giữ nguyên từ baseline.
USE_LORA = True
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["c_attn", "c_proj", "c_fc"]


def ensure_model_token_embeddings(model, tok, pad_id, eos_id):
    """Resize embedding nếu tokenizer có token id ngoài vocab của checkpoint."""
    token_count = len(tok)
    embed_count = model.get_input_embeddings().num_embeddings
    if token_count > embed_count:
        print(f"Resize token embeddings: {embed_count} -> {token_count}")
        model.resize_token_embeddings(token_count)
        embed_count = model.get_input_embeddings().num_embeddings

    special_ids = [int(x) for x in [pad_id, eos_id] if x is not None]
    max_special_id = max(special_ids) if special_ids else -1
    if max_special_id >= embed_count:
        raise ValueError(
            f"PAD/EOS id ngoài embedding vocab: max_special_id={max_special_id}, "
            f"embedding_size={embed_count}. Hãy kiểm tra tokenizer/model hoặc resize embedding."
        )

    model.config.pad_token_id = int(pad_id) if pad_id is not None else None
    model.config.eos_token_id = int(eos_id) if eos_id is not None else None
    return model


def audit_dataset_batch(dataset, data_collator, vocab_size, name):
    if dataset is None or len(dataset) == 0:
        return
    sample_size = min(8, len(dataset))
    batch = [dataset[i] for i in range(sample_size)]
    tensors = data_collator(batch)
    input_ids = tensors["input_ids"]
    labels = tensors["labels"]
    input_min = int(input_ids.min().item())
    input_max = int(input_ids.max().item())
    valid_labels = labels[labels != -100]
    label_min = int(valid_labels.min().item()) if valid_labels.numel() else -100
    label_max = int(valid_labels.max().item()) if valid_labels.numel() else -100
    if input_min < 0 or input_max >= vocab_size or label_max >= vocab_size:
        raise ValueError(
            f"{name} batch có token id ngoài vocab: "
            f"input_range=[{input_min}, {input_max}], "
            f"label_range=[{label_min}, {label_max}], vocab_size={vocab_size}"
        )
    print(
        f"{name} token audit OK | input_range=[{input_min}, {input_max}] "
        f"| label_range=[{label_min}, {label_max}] | vocab_size={vocab_size}"
    )


def print_trainable_parameters(model):
    trainable = 0
    total = 0
    for _, param in model.named_parameters():
        total += param.numel()
        if param.requires_grad:
            trainable += param.numel()
    pct = 100 * trainable / max(1, total)
    print(f"Trainable params: {trainable:,} / {total:,} ({pct:.4f}%)")


tmp_model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
tmp_model = ensure_model_token_embeddings(tmp_model, tokenizer, PAD_ID, EOS_ID)
MODEL_VOCAB_SIZE = tmp_model.get_input_embeddings().num_embeddings
del tmp_model
gc.collect()
torch.cuda.empty_cache()


def stratified_eval_sample_by_type(records, n, seed=SEED, type_key="type"):
    records = [r for r in records if r.get("response_vi")]
    if not records or n is None or n <= 0:
        return []
    n = min(int(n), len(records))
    rng = random.Random(seed)
    groups = defaultdict(list)
    for r in records:
        groups[str(r.get(type_key, "unknown"))].append(r)
    for rec_type in groups:
        rng.shuffle(groups[rec_type])
    selected = []
    type_names = sorted(groups.keys())
    if n >= len(type_names):
        for rec_type in type_names:
            selected.append(groups[rec_type].pop())
    remaining_pool = []
    for rec_type in type_names:
        remaining_pool.extend(groups[rec_type])
    rng.shuffle(remaining_pool)
    selected.extend(remaining_pool[: n - len(selected)])
    rng.shuffle(selected)
    return selected


def _normalize_stage_ratios(ratios):
    total = float(sum(ratios.values()))
    if total <= 0:
        raise ValueError("Stage ratios must sum to a positive number")
    return {str(k): float(v) / total for k, v in ratios.items() if float(v) > 0}


def _stable_record_order(records, stage_name, rec_type):
    keyed = []
    for rec in records:
        key = normalized_hash(
            f"{stage_name}|{rec_type}|{rec.get('id', '')}|{rec.get('query_vi', '')}|{rec.get('response_vi', '')}"
        )
        keyed.append((key, rec))
    keyed.sort(key=lambda x: x[0])
    return [rec for _, rec in keyed]


def _allocate_type_quotas(target_total, ratios, available_by_type, sample_with_replacement=False):
    ratios = _normalize_stage_ratios(ratios)
    target_total = int(target_total)
    raw = {t: target_total * ratio for t, ratio in ratios.items()}
    quotas = {t: int(math.floor(raw[t])) for t in ratios}
    for rec_type in ratios:
        if target_total >= len(ratios) and quotas[rec_type] == 0:
            quotas[rec_type] = 1
    while sum(quotas.values()) < target_total:
        candidates = list(ratios.keys())
        if not sample_with_replacement:
            candidates = [t for t in candidates if quotas[t] < available_by_type.get(t, 0)]
        if not candidates:
            break
        chosen = max(candidates, key=lambda t: (raw[t] - math.floor(raw[t]), ratios[t]))
        quotas[chosen] += 1
    while sum(quotas.values()) > target_total:
        candidates = [t for t, q in quotas.items() if q > 0]
        chosen = min(candidates, key=lambda t: (raw[t] - math.floor(raw[t]), quotas[t]))
        quotas[chosen] -= 1
    if not sample_with_replacement:
        quotas = {t: min(q, available_by_type.get(t, 0)) for t, q in quotas.items()}
    return quotas


def _max_total_without_replacement(ratios, available_by_type):
    ratios = _normalize_stage_ratios(ratios)
    limits = []
    for rec_type, ratio in ratios.items():
        available = available_by_type.get(rec_type, 0)
        if available <= 0:
            raise ValueError(f"Không có sample cho type bắt buộc của curriculum stage: {rec_type}")
        limits.append(math.floor(available / max(ratio, CURRICULUM_RATIO_TOLERANCE)))
    return int(max(1, min(limits)))


def build_curriculum_stage_records(records, stage_spec, max_samples=None, sample_with_replacement=False):
    stage_name = stage_spec["name"]
    ratios = _normalize_stage_ratios(stage_spec["ratios"])
    allowed_types = set(stage_spec["types"])
    groups = defaultdict(list)
    for rec in records:
        rec_type = str(rec.get("type", "unknown"))
        if rec_type in allowed_types:
            groups[rec_type].append(rec)
    available_by_type = {t: len(groups.get(t, [])) for t in ratios}
    missing = [t for t, n in available_by_type.items() if n <= 0]
    if missing:
        raise ValueError(f"Stage {stage_name} thiếu type: {missing}")
    if max_samples is None:
        target_total = sum(available_by_type.values()) if sample_with_replacement else _max_total_without_replacement(ratios, available_by_type)
    else:
        target_total = int(max_samples)
        if not sample_with_replacement:
            target_total = min(target_total, _max_total_without_replacement(ratios, available_by_type))
    quotas = _allocate_type_quotas(target_total, ratios, available_by_type, sample_with_replacement)
    selected = []
    selection_detail = {}
    for rec_type in stage_spec["types"]:
        quota = int(quotas.get(rec_type, 0))
        ordered = _stable_record_order(groups.get(rec_type, []), stage_name, rec_type)
        if sample_with_replacement and quota > len(ordered):
            repeats = math.ceil(quota / max(1, len(ordered)))
            chosen = (ordered * repeats)[:quota]
        else:
            chosen = ordered[:quota]
        selected.extend(chosen)
        selection_detail[rec_type] = {
            "available": len(groups.get(rec_type, [])),
            "selected": len(chosen),
            "target_ratio": ratios.get(rec_type, 0.0),
        }
    selected = _stable_record_order(selected, stage_name, "final_shuffle")
    final_counts = Counter(str(r.get("type", "unknown")) for r in selected)
    for rec_type in selection_detail:
        selection_detail[rec_type]["actual_ratio"] = final_counts.get(rec_type, 0) / max(1, len(selected))
    return selected, {
        "stage_name": stage_name,
        "target_total": target_total,
        "actual_total": len(selected),
        "sample_with_replacement": bool(sample_with_replacement),
        "ratios": ratios,
        "available_by_type": available_by_type,
        "selected_by_type": dict(final_counts),
        "selection_detail": selection_detail,
    }


def _record_total_tokens(rec):
    value = rec.get("total_tokens")
    if value is not None:
        try:
            return int(value)
        except Exception:
            pass
    return int(measure_record_tokens(rec)[2])


def summarize_stage_training_records(stage_name, records, available_counts, effective_batch, epochs, max_steps):
    counts = Counter(str(r.get("type", "unknown")) for r in records)
    token_sums = defaultdict(int)
    for rec in records:
        token_sums[str(rec.get("type", "unknown"))] += _record_total_tokens(rec)
    if max_steps is not None and int(max_steps) > 0:
        update_steps = int(max_steps)
    else:
        update_steps = int(math.ceil(len(records) / max(1, effective_batch)) * float(epochs))
    return {
        "train_samples_by_type": dict(counts),
        "train_tokens_by_type": dict(token_sums),
        "avg_length_by_type": {t: (token_sums[t] / max(1, counts[t])) for t in sorted(counts)},
        "effective_sampling_ratio_by_type": {
            t: (counts.get(t, 0) / max(1, available_counts.get(t, 0)))
            for t in sorted(available_counts)
        },
        "steps_per_type": {
            t: update_steps * counts.get(t, 0) / max(1, len(records))
            for t in sorted(counts)
        },
        "estimated_update_steps": update_steps,
        "effective_batch": effective_batch,
    }


eval_records_for_trainer = stratified_eval_sample_by_type(valid_records, TRAINER_EVAL_SAMPLES, seed=SEED)
eval_ds = MathDataset(eval_records_for_trainer, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH) if eval_records_for_trainer else None
collator = PadCollator(pad_id=PAD_ID)

print("Eval trainer type distribution:")
print(Counter(r.get("type", "unknown") for r in eval_records_for_trainer))

effective_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count() if CUDA_OK else 0)
AVAILABLE_TRAIN_COUNTS_BY_TYPE = Counter(str(r.get("type", "unknown")) for r in train_records)
CURRICULUM_STAGE_RECORDS = {}
CURRICULUM_STAGE_DATASETS = {}
CURRICULUM_STAGE_SELECTION_REPORTS = {}
CURRICULUM_STAGE_TRAIN_STATS = {}
CURRICULUM_STAGE_OUTPUT_DIRS = {}
CURRICULUM_STAGE_VALID_OUTPUT_PATHS = {}
CURRICULUM_STAGE_VALID_REPORT_PATHS = {}
CURRICULUM_STAGE_EVAL_ROWS_PATHS = {}
CURRICULUM_STAGE_EVAL_ROWS_JSON_PATHS = {}

for stage_spec in CURRICULUM_STAGE_SPECS:
    stage_name = stage_spec["name"]
    stage_records, selection_report = build_curriculum_stage_records(
        train_records,
        stage_spec,
        max_samples=CURRICULUM_STAGE_MAX_SAMPLES.get(stage_name),
        sample_with_replacement=CURRICULUM_SAMPLE_WITH_REPLACEMENT,
    )
    stage_ds = MathDataset(stage_records, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH)
    stage_epochs = float(CURRICULUM_STAGE_EPOCHS.get(stage_name, EPOCHS))
    stage_max_steps = CURRICULUM_STAGE_MAX_STEPS.get(stage_name)
    CURRICULUM_STAGE_RECORDS[stage_name] = stage_records
    CURRICULUM_STAGE_DATASETS[stage_name] = stage_ds
    CURRICULUM_STAGE_SELECTION_REPORTS[stage_name] = selection_report
    CURRICULUM_STAGE_TRAIN_STATS[stage_name] = summarize_stage_training_records(
        stage_name, stage_records, AVAILABLE_TRAIN_COUNTS_BY_TYPE, effective_batch, stage_epochs, stage_max_steps
    )
    CURRICULUM_STAGE_OUTPUT_DIRS[stage_name] = STAGE_OUTPUT_ROOT / stage_name
    CURRICULUM_STAGE_VALID_OUTPUT_PATHS[stage_name] = STAGE_PREDICTION_DIR / f"{stage_name}_valid_output.json"
    CURRICULUM_STAGE_VALID_REPORT_PATHS[stage_name] = STAGE_REPORT_DIR / f"{stage_name}_report.json"
    CURRICULUM_STAGE_EVAL_ROWS_PATHS[stage_name] = STAGE_REPORT_DIR / f"{stage_name}_eval_rows.jsonl"
    CURRICULUM_STAGE_EVAL_ROWS_JSON_PATHS[stage_name] = STAGE_REPORT_DIR / f"{stage_name}_eval_rows.json"
    audit_dataset_batch(stage_ds, collator, MODEL_VOCAB_SIZE, stage_name)

train_ds = CURRICULUM_STAGE_DATASETS["stage4_all"]
OUTPUT_DIR = CURRICULUM_STAGE_OUTPUT_DIRS["stage4_all"]
steps_per_epoch = math.ceil(len(train_ds) / effective_batch)
total_train_steps = sum(CURRICULUM_STAGE_TRAIN_STATS[s["name"]]["estimated_update_steps"] for s in CURRICULUM_STAGE_SPECS)
WARMUP_STEPS = max(1, int(total_train_steps * WARMUP_RATIO)) if total_train_steps > 0 else 0

audit_dataset_batch(eval_ds, collator, MODEL_VOCAB_SIZE, "eval")
print("train source:", globals().get("TRAIN_SOURCE", "train_records"))
print("preprocessed train file:", PREPROCESSED_TRAIN_FILE)
print("available train samples by type:")
print(json.dumps(dict(AVAILABLE_TRAIN_COUNTS_BY_TYPE), ensure_ascii=False, indent=2))
print("effective batch:", effective_batch)
print("curriculum estimated total update steps:", total_train_steps)
print("stage selection reports:")
print(json.dumps(CURRICULUM_STAGE_SELECTION_REPORTS, ensure_ascii=False, indent=2)[:6000])


def make_training_args(stage_name, output_dir, train_dataset, num_train_epochs, warmup_steps, max_steps=None):
    use_bf16 = bool(CUDA_OK and torch.cuda.is_bf16_supported())
    use_fp16 = bool(CUDA_OK and not use_bf16)
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=float(num_train_epochs),
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_steps=int(warmup_steps),
        lr_scheduler_type="cosine",
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        logging_steps=LOGGING_STEPS,
        save_strategy="epoch" if SAVE_DURING_TRAIN else "no",
        save_total_limit=1,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        dataloader_num_workers=4 if IS_KAGGLE else 0,
        gradient_checkpointing=False,
    )
    if max_steps is not None and int(max_steps) > 0:
        kwargs["max_steps"] = int(max_steps)
    sig = inspect.signature(TrainingArguments.__init__)
    has_eval = eval_ds is not None
    eval_strategy_value = "epoch" if (has_eval and EVAL_DURING_TRAIN) else "no"
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = eval_strategy_value
    else:
        kwargs["evaluation_strategy"] = eval_strategy_value
    if "bf16" in sig.parameters:
        kwargs["bf16"] = use_bf16
    if "fp16" in sig.parameters:
        kwargs["fp16"] = use_fp16
    if "optim" in sig.parameters and CUDA_OK:
        kwargs["optim"] = "adamw_torch_fused"
    if not CUDA_OK:
        if "use_cpu" in sig.parameters:
            kwargs["use_cpu"] = True
        elif "no_cuda" in sig.parameters:
            kwargs["no_cuda"] = True
    return TrainingArguments(**kwargs)


In [ ]:
# 10. Audit target sau preprocessing trước khi train
RESPONSE_AUDIT_PATH = WORK_DIR / "response_audit_report.json"
AUDIT_SAMPLES_PER_TYPE = 20
AUDIT_TOKEN_SAMPLE_SIZE = min(2000, len(train_records))
PLACEHOLDER_RATE_THRESHOLD = 0.02
MIN_CALC_RATE_WARNING = 0.35

PLACEHOLDER_PATTERNS = [
    r"Tính theo dữ kiện trong đề",
    r"Tính theo dữ liệu trong đề",
    r"Dựa vào dữ kiện trong đề",
    r"Lời giải:\s*(?:Tính theo dữ kiện trong đề\.?\s*)?Đáp án là",
]

CALC_PATTERNS = [
    r"\d\s*(?:\+|-|\*|/|=|×|÷)\s*\d",
    r"(?:cộng|trừ|nhân|chia|bằng|tổng|hiệu|tích|thương|suy ra|ta có)",
    r"\$[^$]*(?:\+|-|=|\\frac|\\times|\\div)[^$]*\$",
]

ANSWER_AT_END_RE = re.compile(
    # r"\nĐáp án là\s*[:：]?\s*([^\n]+)\s*$",
    r"(?:^|\n)\s*Đáp án là\s*[:：]?\s*([^\n]+)\s*$",
    flags=re.IGNORECASE
)


def is_placeholder_response(text):
    text = str(text or "")
    return any(re.search(p, text, flags=re.IGNORECASE) for p in PLACEHOLDER_PATTERNS)


def has_intermediate_calculation(text):
    text = str(text or "")
    body = ANSWER_AT_END_RE.sub("", text)
    return any(re.search(p, body, flags=re.IGNORECASE) for p in CALC_PATTERNS)


def final_answer_at_end(rec):
    text = str(rec.get("response_vi", ""))
    m = ANSWER_AT_END_RE.search(text)
    if not m:
        return False
    gold = str(rec.get("answer_text") or "").strip()
    return (not gold) or (gold in m.group(1).strip())


def record_response_audit_row(rec):
    response = str(build_target_response(rec))
    final_match = ANSWER_AT_END_RE.search(response)
    gold = str(rec.get("answer_text") or "").strip()
    final_answer_ok = bool(final_match) and ((not gold) or (gold in final_match.group(1).strip()))
    return {
        "id": rec.get("id"),
        "type": rec.get("type", "unknown"),
        "task_group": get_task_group(rec.get("type", "unknown")),
        "target_mode": target_mode_for_record(rec),
        "response_words": word_count(response),
        "response_tokens": len(encode_no_special(response)) + 1,
        "was_truncated": bool(rec.get("was_truncated")),
        "placeholder_like": is_placeholder_response(response),
        "has_intermediate_calculation": has_intermediate_calculation(response),
        "final_answer_at_end": final_answer_ok,
        "answer_text": rec.get("answer_text"),
        "response_preview": response[:500],
    }


audit_rows = [record_response_audit_row(r) for r in train_records]
audit_df = pd.DataFrame(audit_rows)

# Label density audit: tỷ lệ token thực sự được dùng để train (labels != -100)
token_sample_indices = list(range(len(train_records)))
if len(token_sample_indices) > AUDIT_TOKEN_SAMPLE_SIZE:
    token_sample_indices = random.sample(token_sample_indices, AUDIT_TOKEN_SAMPLE_SIZE)

label_rows = []
for idx in tqdm(token_sample_indices, desc="label density audit"):
    item = train_ds[idx]
    total = int(sum(item["attention_mask"]))
    labeled = int(sum(1 for x in item["labels"] if x != -100))
    rec = train_records[idx]
    label_rows.append({
        "id": rec.get("id"),
        "type": rec.get("type", "unknown"),
        "task_group": get_task_group(rec.get("type", "unknown")),
        "labeled_tokens": labeled,
        "total_tokens": total,
        "label_token_ratio": labeled / max(1, total),
    })

label_df = pd.DataFrame(label_rows)
summary_df = (
    audit_df.merge(label_df[["id", "label_token_ratio"]], on="id", how="left")
    .groupby(["type", "task_group"], dropna=False)
    .agg(
        n=("id", "count"),
        response_words_p50=("response_words", "median"),
        response_words_p95=("response_words", lambda s: s.quantile(0.95)),
        response_tokens_p50=("response_tokens", "median"),
        response_tokens_p95=("response_tokens", lambda s: s.quantile(0.95)),
        label_ratio_mean=("label_token_ratio", "mean"),
        label_ratio_p05=("label_token_ratio", lambda s: s.quantile(0.05)),
        placeholder_rate=("placeholder_like", "mean"),
        answer_only_rate=("target_mode", lambda s: float((s == "answer_only").mean())),
        calc_rate=("has_intermediate_calculation", "mean"),
        truncated_rate=("was_truncated", "mean"),
        final_answer_missing_or_moved_rate=("final_answer_at_end", lambda s: 1 - s.mean()),
    )
    .reset_index()
)

print("Response/target audit summary:")
display(summary_df.round(4))

print("\nMẫu response_vi theo từng type sau preprocessing:")
for t in sorted(audit_df["type"].dropna().unique()):
    print("\n" + "=" * 90)
    print(f"TYPE: {t} | TASK: {get_task_group(t)}")
    type_rows = audit_df[audit_df["type"] == t]
    n_show = AUDIT_SAMPLES_PER_TYPE + (2 if "SV" in t or "FOBAR" in t else 0)
    for _, row in type_rows.head(n_show).iterrows():
        print("-" * 90)
        print(f"id={row['id']} | words={row['response_words']} | placeholder={row['placeholder_like']} | calc={row['has_intermediate_calculation']} | truncated={row['was_truncated']} | final_at_end={row['final_answer_at_end']}")
        print(row["response_preview"])

placeholder_rate = float(audit_df["placeholder_like"].mean()) if len(audit_df) else 1.0
calc_rate = float(audit_df["has_intermediate_calculation"].mean()) if len(audit_df) else 0.0
final_missing_rate = float((~audit_df["final_answer_at_end"]).mean()) if len(audit_df) else 1.0
truncated_rate = float(audit_df["was_truncated"].mean()) if len(audit_df) else 0.0
answer_only_rate = float((audit_df["target_mode"] == "answer_only").mean()) if len(audit_df) else 0.0
min_label_ratio = float(label_df["label_token_ratio"].min()) if len(label_df) else 0.0
mean_label_ratio = float(label_df["label_token_ratio"].mean()) if len(label_df) else 0.0

TARGET_READY_FOR_TRAIN = (
    placeholder_rate <= PLACEHOLDER_RATE_THRESHOLD
    and final_missing_rate == 0.0
    and min_label_ratio > 0.0
)

response_audit_report = {
    "summary_by_type": summary_df.to_dict("records"),
    "overall": {
        "n": len(audit_rows),
        "target_mode": TARGET_MODE,
        "answer_only_ratio": ANSWER_ONLY_RATIO,
        "answer_only_rate": answer_only_rate,
        "placeholder_rate": placeholder_rate,
        "calc_rate": calc_rate,
        "final_answer_missing_or_moved_rate": final_missing_rate,
        "truncated_rate": truncated_rate,
        "label_ratio_mean": mean_label_ratio,
        "label_ratio_min_sampled": min_label_ratio,
        "target_ready_for_train": TARGET_READY_FOR_TRAIN,
    },
    "sample_rows": audit_rows[:80],
    "label_density_sample": label_rows[:200],
}
save_json(response_audit_report, RESPONSE_AUDIT_PATH)
print("\nWrote:", RESPONSE_AUDIT_PATH)

if placeholder_rate > PLACEHOLDER_RATE_THRESHOLD:
    print(f"WARNING: placeholder_like rate = {placeholder_rate:.2%}. Cần khôi phục/generate lại reasoning target trước khi train.")
if calc_rate < MIN_CALC_RATE_WARNING:
    print(f"WARNING: calc_rate = {calc_rate:.2%}. Response có thể thiếu lời giải trung gian thật.")
if final_missing_rate > 0:
    print(f"WARNING: {final_missing_rate:.2%} response không có final answer ở cuối; kiểm tra smart truncation trước khi train.")
if min_label_ratio <= 0:
    print("WARNING: Có sample trong audit có label_token_ratio = 0; đây là lỗi supervision nghiêm trọng.")

print("TARGET_READY_FOR_TRAIN:", TARGET_READY_FOR_TRAIN)


In [ ]:
# 10. Train curriculum 4 stage và lưu checkpoint từng stage
CURRICULUM_STAGE_TRAIN_METRICS = {}

if RUN_TRAIN:
    if not globals().get("TARGET_READY_FOR_TRAIN", False):
        raise RuntimeError(
            "TARGET_READY_FOR_TRAIN=False. Hãy xử lý response_vi placeholder/truncation/label audit trước khi train."
        )
    if not CUDA_OK:
        raise RuntimeError(
            "Không có GPU CUDA dùng được cho full training. Hãy đổi Accelerator sang T4/V100/A100."
        )

    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = ensure_model_token_embeddings(model, tokenizer, PAD_ID, EOS_ID)
    model.config.use_cache = False

    if USE_LORA:
        if not PEFT_AVAILABLE:
            raise RuntimeError("USE_LORA=True nhưng peft chưa import được. Hãy cài/enable peft trước khi chạy.")
        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            target_modules=LORA_TARGET_MODULES,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            task_type="CAUSAL_LM",
            fan_in_fan_out=True,
        )
        model = get_peft_model(model, lora_config)
        print("Using LoRA config:")
        print(lora_config)

    print_trainable_parameters(model)
    curriculum_start = time.time()

    for stage_index, stage_spec in enumerate(CURRICULUM_STAGE_SPECS, start=1):
        stage_name = stage_spec["name"]
        stage_ds = CURRICULUM_STAGE_DATASETS[stage_name]
        stage_output_dir = CURRICULUM_STAGE_OUTPUT_DIRS[stage_name]
        stage_epochs = float(CURRICULUM_STAGE_EPOCHS.get(stage_name, EPOCHS))
        stage_max_steps = CURRICULUM_STAGE_MAX_STEPS.get(stage_name)
        stage_steps = CURRICULUM_STAGE_TRAIN_STATS[stage_name]["estimated_update_steps"]
        stage_warmup_steps = max(1, int(stage_steps * WARMUP_RATIO)) if stage_steps > 0 else 0

        print("\n" + "=" * 100)
        print(f"Curriculum stage {stage_index}/{len(CURRICULUM_STAGE_SPECS)}: {stage_name}")
        print("types:", stage_spec["types"])
        print("ratios:", stage_spec["ratios"])
        print("samples:", len(stage_ds), "| epochs:", stage_epochs, "| max_steps:", stage_max_steps)
        print("estimated update steps:", stage_steps, "| warmup steps:", stage_warmup_steps)
        print("selected_by_type:", CURRICULUM_STAGE_SELECTION_REPORTS[stage_name]["selected_by_type"])

        trainer = Trainer(
            model=model,
            args=make_training_args(stage_name, stage_output_dir, stage_ds, stage_epochs, stage_warmup_steps, max_steps=stage_max_steps),
            train_dataset=stage_ds,
            eval_dataset=eval_ds,
            data_collator=collator,
        )

        start = time.time()
        train_output = trainer.train()
        train_minutes = round((time.time() - start) / 60, 2)
        print(f"[{stage_name}] train minutes:", train_minutes)
        print(train_output)

        eval_metrics = {}
        if EVAL_LOSS_AFTER_EACH_STAGE and eval_ds is not None:
            eval_metrics = trainer.evaluate(eval_dataset=eval_ds, metric_key_prefix=f"{stage_name}_eval")
            print(f"[{stage_name}] eval loss metrics:")
            print(json.dumps(eval_metrics, ensure_ascii=False, indent=2))

        trainer.save_model(str(stage_output_dir))
        tokenizer.save_pretrained(str(stage_output_dir))
        print(f"[{stage_name}] saved checkpoint:", stage_output_dir)

        CURRICULUM_STAGE_TRAIN_METRICS[stage_name] = {
            "stage_index": stage_index,
            "stage_name": stage_name,
            "train_minutes": train_minutes,
            "train_metrics": dict(train_output.metrics),
            "eval_metrics": eval_metrics,
            "output_dir": str(stage_output_dir),
            "selection_report": CURRICULUM_STAGE_SELECTION_REPORTS[stage_name],
            "train_stats": CURRICULUM_STAGE_TRAIN_STATS[stage_name],
        }

        del trainer
        gc.collect()
        torch.cuda.empty_cache()

    print("\nCurriculum total train minutes:", round((time.time() - curriculum_start) / 60, 2))
    OUTPUT_DIR = CURRICULUM_STAGE_OUTPUT_DIRS["stage4_all"]
    del model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Skip train. Inference sẽ dùng checkpoint stage nếu có, nếu không dùng base model.")


In [ ]:
# 11. Hàm sinh lời giải V2 — batch + beam search + custom StoppingCriteria
if TARGET_MODE == "answer_only":
    MAX_NEW_TOKENS = 64
    REPETITION_PENALTY = 1.1
    NO_REPEAT_NGRAM_SIZE = 3
elif TARGET_MODE == "mixed" and ANSWER_ONLY_RATIO >= 0.5:
    MAX_NEW_TOKENS = 128
    REPETITION_PENALTY = 1.2
    NO_REPEAT_NGRAM_SIZE = 3
else:
    MAX_NEW_TOKENS = 256
    REPETITION_PENALTY = 1.3
    NO_REPEAT_NGRAM_SIZE = 4

NUM_BEAMS = 2
DO_SAMPLE = False
INFER_BATCH_SIZE = 24
EARLY_STOPPING = True

BASELINE_SOURCE = VARIANT_NAME
BASELINE_DECODING = f"beam2_{TARGET_MODE}_antiloop_v4"
# BASELINE_DECODING = "beam" if NUM_BEAMS > 1 else "greedy"


class AnswerStoppingCriteria(StoppingCriteria):
    """
    Dừng generation khi model đã sinh một answer anchor + số đầu tiên.
    Không đợi model tự kết thúc vì GPT-2 hiện dễ sinh đuôi rác sau đáp án.
    """
    def __init__(self, tokenizer, prompt_lens, eos_id, check_every=4):
        super().__init__()
        self.tokenizer = tokenizer
        self.prompt_lens = prompt_lens
        self.eos_id = eos_id
        self.pattern = re.compile(
            r"(?:Đáp\s*án\s*là|Đáp\s*án|Câu\s*trả\s*lời\s*là|Kết\s*quả\s*là|Answer|The answer is)\s*[:：]?\s*[\[\{\(]?\s*[-+]?\d",
            re.IGNORECASE,
        )
        self.check_every = check_every
        self._step = 0

    def __call__(self, input_ids, scores, **kwargs):
        self._step += 1
        if self._step % self.check_every != 0:
            return False

        done = []
        for i in range(input_ids.shape[0]):
            tail = input_ids[i][-120:].tolist()
            text = self.tokenizer.decode(tail, skip_special_tokens=True)
            done.append(bool(self.pattern.search(text)))

        return all(done)
        

def save_json(obj, path):
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def postprocess_output(text):
    """
    Chuẩn hóa output về:
    <reasoning trước anchor>
    Đáp án là: <số đầu tiên sau anchor>

    Mục tiêu: loại bỏ đuôi rác sau đáp án, không để evaluator lấy số cuối sai.
    """
    text = str(text or "").strip()

    # Cắt các marker bắt đầu sample/prompt mới nếu model lặp prompt.
    for marker in [
        "\nCâu hỏi:",
        "\nQuestion:",
        "\nBài toán:",
        "\n[Loại:",
        "\n###",
        "\nGiải bài toán",
        "\nLoại bài:",
        "\n[TASK:",
    ]:
        pos = text.find(marker)
        if pos >= 0:
            text = text[:pos].strip()

    matches = list(ANSWER_ANCHOR_RE.finditer(text))
    if matches:
        m = matches[0]  # với model output: dùng anchor đầu tiên
        reasoning = text[:m.start()].strip()
        tail = text[m.end():]
        ans = first_answer_unit(tail)

        if ans is not None:
            if reasoning:
                return reasoning + f"\nĐáp án là: {ans}"
            return f"Đáp án là: {ans}"

        # Nếu có anchor nhưng chưa đọc được số, chỉ giữ ngắn lại.
        return text[:m.end() + 80].strip()

    return text


def ensure_anchor(text):
    """
    Nếu model không sinh anchor nhưng có số cuối thì gắn anchor.
    Đây chỉ là fallback; evaluation chính vẫn ưu tiên số đầu tiên sau anchor.
    """
    text = str(text or "").strip()

    if ANSWER_ANCHOR_RE.search(text):
        return postprocess_output(text)

    last_num = extract_answer_text(text, allow_last_number=True)
    if last_num:
        return text.rstrip() + f"\nĐáp án là: {last_num}"

    return text
    

def load_model_for_generation(model_dir, tokenizer, dtype, device):
    model_dir = Path(model_dir)
    adapter_config = model_dir / "adapter_config.json"

    if adapter_config.exists():
        if not PEFT_AVAILABLE:
            raise RuntimeError("Checkpoint là LoRA adapter nhưng peft chưa import được.")

        print("Detected LoRA adapter checkpoint:", model_dir)
        base_model = AutoModelForCausalLM.from_pretrained(
            str(MODEL_DIR),
            torch_dtype=dtype,
            local_files_only=True,
        )
        base_model = ensure_model_token_embeddings(base_model, tokenizer, PAD_ID, EOS_ID)
        model = PeftModel.from_pretrained(
            base_model,
            str(model_dir),
            local_files_only=True,
        )
    else:
        print("Detected full model checkpoint:", model_dir)
        model = AutoModelForCausalLM.from_pretrained(
            str(model_dir),
            torch_dtype=dtype,
            local_files_only=True,
        )
        model = ensure_model_token_embeddings(model, tokenizer, PAD_ID, EOS_ID)

    model = model.to(device)
    model.config.use_cache = True
    model.eval()
    return model


def generate_predictions(model_dir, records, output_path, name):
    model_dir = Path(model_dir)
    output_path = Path(output_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if device.type == "cuda" else torch.float32

    gen_tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)

    gen_tokenizer.pad_token_id = PAD_ID
    gen_tokenizer.eos_token_id = EOS_ID
    gen_tokenizer.padding_side = "left"

    if getattr(gen_tokenizer, "pad_token", None) is None:
        if getattr(gen_tokenizer, "eos_token", None) is not None:
            gen_tokenizer.pad_token = gen_tokenizer.eos_token
        else:
            gen_tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

    model = load_model_for_generation(model_dir, gen_tokenizer, dtype, device)

    vocab_size = model.get_input_embeddings().num_embeddings
    outputs = []
    start_all = time.time()

    # Sort records by query length để batch tương đối đồng đều
    order = sorted(range(len(records)), key=lambda i: len(records[i].get("query_vi", "")))
    sorted_records = [records[i] for i in order]

    with torch.inference_mode():
        for batch_start in tqdm(range(0, len(sorted_records), INFER_BATCH_SIZE), desc=name):
            batch = sorted_records[batch_start: batch_start + INFER_BATCH_SIZE]
            prompts = [build_prompt(r) for r in batch]
            enc = gen_tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(device)
            input_ids = enc["input_ids"].clamp(min=0, max=vocab_size - 1)
            attention_mask = enc["attention_mask"]
            prompt_lens = attention_mask.sum(dim=1).tolist()

            stopping = StoppingCriteriaList([
                AnswerStoppingCriteria(gen_tokenizer, prompt_lens, EOS_ID, check_every=4)
            ])

            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                num_beams=NUM_BEAMS,
                early_stopping=EARLY_STOPPING,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=PAD_ID,
                eos_token_id=EOS_ID,
                stopping_criteria=stopping,
            )

            # Với left padding, prompt nằm bên trái, generated = từ input_ids.shape[1] trở đi
            prompt_len_tensor = input_ids.shape[1]
            for i, rec in enumerate(batch):
                new_tokens = gen[i, prompt_len_tensor:]
                text = gen_tokenizer.decode(new_tokens, skip_special_tokens=True)
                text = postprocess_output(text)
                text = ensure_anchor(text)
                rec_type = rec.get("type", "unknown")
                outputs.append({
                    "id": rec.get("id"),
                    "query_vi": rec["query_vi"],
                    "type": rec_type,
                    "task_group": get_task_group(rec_type),
                    "source": BASELINE_SOURCE,
                    "decoding": BASELINE_DECODING,
                    "model_output": text,
                    "_sort_index": batch_start + i,
                })

    # Sắp xếp lại theo thứ tự gốc (map qua order)
    inv_order = {sorted_idx: orig_idx for orig_idx, sorted_idx in enumerate(order)}
    outputs_final = [None] * len(outputs)
    for o in outputs:
        sorted_pos = o.pop("_sort_index")
        original_pos = order[sorted_pos]
        outputs_final[original_pos] = o
    outputs_final = [o for o in outputs_final if o is not None]

    save_json(outputs_final, output_path)
    print("Wrote:", output_path)
    print("Minutes:", round((time.time() - start_all) / 60, 2))

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return outputs_final


In [ ]:
# 12. Sinh output validation cho từng curriculum stage
RUN_VALIDATION = True
CURRICULUM_STAGE_VALID_OUTPUTS = {}
MODEL_FOR_INFERENCE = CURRICULUM_STAGE_OUTPUT_DIRS.get("stage4_all", OUTPUT_DIR)

if RUN_VALIDATION and valid_records:
    for stage_spec in CURRICULUM_STAGE_SPECS:
        stage_name = stage_spec["name"]
        stage_model_dir = Path(CURRICULUM_STAGE_OUTPUT_DIRS[stage_name])
        stage_output_path = CURRICULUM_STAGE_VALID_OUTPUT_PATHS[stage_name]

        if not stage_model_dir.exists():
            print(f"Skip validation generation for {stage_name}: checkpoint chưa tồn tại tại {stage_model_dir}")
            CURRICULUM_STAGE_VALID_OUTPUTS[stage_name] = []
            continue

        outputs = generate_predictions(stage_model_dir, valid_records, stage_output_path, name=f"validation:{stage_name}")
        for row in outputs:
            row["stage_name"] = stage_name
        save_json(outputs, stage_output_path)
        CURRICULUM_STAGE_VALID_OUTPUTS[stage_name] = outputs

    valid_outputs = CURRICULUM_STAGE_VALID_OUTPUTS.get("stage4_all", [])
    if valid_outputs:
        save_json(valid_outputs, VALID_OUTPUT_PATH)
        print("\nFinal-stage output mẫu:")
        print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:1200])
else:
    valid_outputs = []
    print("Skip validation generation")


In [ ]:
# 13. Đánh giá validation + lưu valid_report.json
CASES_TO_SHOW = 8
VALID_EVAL_ROWS_PATH = WORK_DIR / "valid_eval_rows.jsonl"
VALID_EVAL_ROWS_JSON_PATH = WORK_DIR / "valid_eval_rows.json"


def extract_query_numbers(text):
    nums = []
    for raw in NUM_RE.findall(str(text or "")):
        val = parse_number(raw)
        if val is not None:
            nums.append(val)
    return nums


def close_number(a, b, rel_tol=1e-9, abs_tol=1e-9):
    if a is None or b is None:
        return False
    return abs(a - b) <= max(abs_tol, rel_tol * max(1.0, abs(a), abs(b)))


def detect_repeated_ngram(text, min_n=4, max_n=10, min_repeat=4):
    words = re.findall(r"\S+", str(text or ""))
    if len(words) < min_n * min_repeat:
        return {
            "has_loop": False,
            "loop_repeat_count": 0,
            "loop_ngram": None,
        }

    best_count = 0
    best_ngram = None

    for n in range(min_n, max_n + 1):
        counts = Counter()
        for i in range(0, len(words) - n + 1):
            ng = tuple(words[i:i+n])
            counts[ng] += 1

        if counts:
            ng, cnt = counts.most_common(1)[0]
            if cnt > best_count:
                best_count = cnt
                best_ngram = " ".join(ng)

    return {
        "has_loop": best_count >= min_repeat,
        "loop_repeat_count": int(best_count),
        "loop_ngram": best_ngram,
    }


def arithmetic_eval_binary(a, op, b):
    if op in ["+", "＋"]:
        return a + b
    if op in ["-", "−"]:
        return a - b
    if op in ["*", "×", "x", "X"]:
        return a * b
    if op in ["/", "÷"]:
        if abs(b) < 1e-12:
            return None
        return a / b
    return None


SIMPLE_EQUATION_RE = re.compile(
    r"([-+]?\d+(?:[.,]\d+)?)\s*"
    r"([+\-−*/×xX÷])\s*"
    r"([-+]?\d+(?:[.,]\d+)?)\s*"
    r"=\s*"
    r"([-+]?\d+(?:[.,]\d+)?)"
)


def verify_simple_arithmetic(text, max_checks=20, tol=1e-6):
    """
    Verifier nhẹ: chỉ kiểm tra các phép nhị phân đơn giản a op b = c.
    Không cố giải toàn bộ reasoning.
    """
    text = str(text or "")
    checks = []
    failed = 0
    passed = 0

    for m in SIMPLE_EQUATION_RE.finditer(text):
        if len(checks) >= max_checks:
            break

        a = parse_number(m.group(1))
        op = m.group(2)
        b = parse_number(m.group(3))
        c = parse_number(m.group(4))

        if a is None or b is None or c is None:
            continue

        expected = arithmetic_eval_binary(a, op, b)
        if expected is None:
            continue

        ok = abs(expected - c) <= max(tol, tol * max(1.0, abs(expected), abs(c)))

        checks.append({
            "expr": m.group(0),
            "expected": expected,
            "actual": c,
            "ok": bool(ok),
        })

        if ok:
            passed += 1
        else:
            failed += 1

    if not checks:
        status = "unknown"
    elif failed > 0:
        status = "fail"
    else:
        status = "pass"

    return {
        "arithmetic_status": status,
        "arithmetic_checked": len(checks),
        "arithmetic_passed": passed,
        "arithmetic_failed": failed,
        "arithmetic_fail_examples": [x for x in checks if not x["ok"]][:5],
    }


def build_verifier_report_for_output(model_output, pred_answer, pred_num):
    text = str(model_output or "")

    loop_info = detect_repeated_ngram(text)
    arith_info = verify_simple_arithmetic(text)

    return {
        "answer_anchor_found": bool(ANSWER_ANCHOR_RE.search(text)),
        "pred_answer_exists": pred_answer is not None,
        "pred_num_exists": pred_num is not None,
        **loop_info,
        **arith_info,
    }
    

def classify_error(score, pred_answer, pred_num, gold_num, query_nums, rec_type, verifier=None):
    verifier = verifier or {}
    if pred_answer is None or pred_num is None:
        return "parse_or_no_number"
    if score == 10:
        return "correct_1pct"
    if score in (5, 1):
        return "near_miss_numeric"
    if verifier.get("has_loop"):
        return "loop"
    if verifier.get("arithmetic_status") == "fail":
        return "arithmetic_inconsistent"
    if any(close_number(pred_num, q) for q in query_nums):
        return "copy_input_number"
    if "SV" in str(rec_type):
        return "wrong_solve_for_variable"
    if "FOBAR" in str(rec_type):
        return "wrong_reverse_param"
    return "wrong_numeric_answer"


def align_by_id(preds, golds):
    if all("id" in x for x in preds) and all("id" in x for x in golds):
        pred_map = {str(x["id"]): x for x in preds}
        return [(pred_map[str(g["id"])], g) for g in golds if str(g["id"]) in pred_map]
    return list(zip(preds, golds))


def evaluate_predictions(preds, golds):
    rows = []
    for row_index, (pred, gold) in enumerate(align_by_id(preds, golds)):
        gold_answer = extract_answer_text(
            gold.get("response_vi"),
            allow_last_number=True,
            prefer_first_anchor=False,
        )
        
        pred_answer = extract_answer_text(
            pred.get("model_output"),
            allow_last_number=False,
            prefer_first_anchor=True,
        )
        
        # fallback defensive: nếu vì lý do nào đó output không có anchor
        if pred_answer is None:
            pred_answer = extract_answer_text(
                pred.get("model_output"),
                allow_last_number=True,
                prefer_first_anchor=True,
            )
        gold_num = parse_number(gold_answer)
        pred_num = parse_number(pred_answer)
        rel_err = relative_error(pred_num, gold_num)
        score = score_one(rel_err, pred_answer is not None)
        rec_type = gold.get("type")
        query_nums = extract_query_numbers(gold.get("query_vi"))
        
        verifier = build_verifier_report_for_output(
            pred.get("model_output"),
            pred_answer,
            pred_num,
        )
        rows.append({
            "row_index": row_index,
            "id": gold.get("id"),
            "type": rec_type,
            "task_group": get_task_group(rec_type),
            "source": pred.get("source", "pure_model"),
            "decoding": pred.get("decoding", BASELINE_DECODING if "BASELINE_DECODING" in globals() else None),
            "query_vi": gold.get("query_vi"),
            "model_output": pred.get("model_output"),
            "gold_answer": gold_answer,
            "pred_answer": pred_answer,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "query_numbers": query_nums,
            "rel_error": rel_err,
            "extractable": pred_answer is not None,
            "score": score,
            "answer_anchor_found": verifier["answer_anchor_found"],
            "has_loop": verifier["has_loop"],
            "loop_repeat_count": verifier["loop_repeat_count"],
            "loop_ngram": verifier["loop_ngram"],
            "arithmetic_status": verifier["arithmetic_status"],
            "arithmetic_checked": verifier["arithmetic_checked"],
            "arithmetic_passed": verifier["arithmetic_passed"],
            "arithmetic_failed": verifier["arithmetic_failed"],
            "arithmetic_fail_examples": verifier["arithmetic_fail_examples"],
            "error_bucket": classify_error(score, pred_answer, pred_num, gold_num, query_nums, rec_type, verifier),
        })
    return rows


def score_summary(rows):
    n = len(rows)
    raw = sum(r["score"] for r in rows)
    return {
        "n": n,
        "raw_score": raw,
        "max_raw_score": 10 * n,
        "score_10": raw / n if n else 0,
        "extractable_rate": sum(r["extractable"] for r in rows) / n if n else 0,
        "buckets": {str(s): sum(r["score"] == s for r in rows) for s in [10, 5, 1, 0]},
    }


def show_cases(title, rows):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    if not rows:
        print("Không có case")
        return
    cols = ["row_index", "id", "type", "score", "rel_error", "gold_answer", "pred_answer", "query_vi", "model_output"]
    display(pd.DataFrame(rows[:CASES_TO_SHOW])[cols])


def _score_10_by_type(eval_df):
    if eval_df.empty:
        return {}
    return {str(t): float(g["score"].mean()) for t, g in eval_df.groupby("type")}


def _bucket_count_by_type(eval_df, score_value):
    if eval_df.empty:
        return {}
    rows = eval_df[eval_df["score"] == score_value]
    return {str(k): int(v) for k, v in rows.groupby("type").size().to_dict().items()}


def _bucket_rate_by_type(eval_df, score_value):
    if eval_df.empty:
        return {}
    return {str(t): float((g["score"] == score_value).mean()) for t, g in eval_df.groupby("type")}


def _prefix_score(eval_df, prefix):
    if eval_df.empty:
        return None
    sub = eval_df[eval_df["type"].astype(str).str.startswith(prefix)]
    if sub.empty:
        return None
    return float(sub["score"].mean())


def _error_bucket_summary(eval_df, bucket_name):
    if eval_df.empty:
        return {"total": 0, "by_type": {}}
    sub = eval_df[eval_df["error_bucket"] == bucket_name]
    return {
        "total": int(len(sub)),
        "by_type": {str(k): int(v) for k, v in sub.groupby("type").size().to_dict().items()},
    }


def build_stage_eval_report(stage_name, stage_outputs):
    eval_rows = evaluate_predictions(stage_outputs, valid_records) if stage_outputs else []
    summary = score_summary(eval_rows)
    eval_df = pd.DataFrame(eval_rows)

    if not eval_df.empty:
        by_type = eval_df.groupby("type").apply(lambda x: pd.Series(score_summary(x.to_dict("records")))).reset_index()
        error_by_type = (
            eval_df.groupby(["type", "task_group", "error_bucket"])
            .size()
            .reset_index(name="count")
            .sort_values(["type", "count"], ascending=[True, False])
        )
        verifier_summary = {
            "anchor_found_rate": float(eval_df["answer_anchor_found"].mean()),
            "loop_rate": float(eval_df["has_loop"].mean()),
            "arithmetic_checked_rate": float((eval_df["arithmetic_checked"] > 0).mean()),
            "arithmetic_fail_rate_all": float((eval_df["arithmetic_status"] == "fail").mean()),
            "arithmetic_fail_rate_checked": float(
                ((eval_df["arithmetic_status"] == "fail") & (eval_df["arithmetic_checked"] > 0)).sum()
                / max(1, (eval_df["arithmetic_checked"] > 0).sum())
            ),
        }
        verifier_by_type = (
            eval_df.groupby(["type", "task_group"])
            .agg(
                n=("type", "size"),
                loop_rate=("has_loop", "mean"),
                anchor_found_rate=("answer_anchor_found", "mean"),
                arithmetic_checked_rate=("arithmetic_checked", lambda x: float((x > 0).mean())),
                arithmetic_fail_rate=("arithmetic_status", lambda x: float((x == "fail").mean())),
            )
            .reset_index()
        )
    else:
        by_type = pd.DataFrame()
        error_by_type = pd.DataFrame()
        verifier_by_type = pd.DataFrame()
        verifier_summary = {
            "anchor_found_rate": 0.0,
            "loop_rate": 0.0,
            "arithmetic_checked_rate": 0.0,
            "arithmetic_fail_rate_all": 0.0,
            "arithmetic_fail_rate_checked": 0.0,
        }

    report = {
        "stage_name": stage_name,
        "summary": summary,
        "overall_score_10": summary["score_10"],
        "overall": {"score_10": summary["score_10"]},
        "score_10_by_type": _score_10_by_type(eval_df),
        "GSM_all_score": _prefix_score(eval_df, "GSM"),
        "MATH_all_score": _prefix_score(eval_df, "MATH"),
        "bucket_10_by_type": _bucket_count_by_type(eval_df, 10),
        "bucket_0_by_type": _bucket_count_by_type(eval_df, 0),
        "bucket_10_rate_by_type": _bucket_rate_by_type(eval_df, 10),
        "bucket_0_rate_by_type": _bucket_rate_by_type(eval_df, 0),
        "arithmetic_inconsistent": _error_bucket_summary(eval_df, "arithmetic_inconsistent"),
        "wrong_solve_for_variable": _error_bucket_summary(eval_df, "wrong_solve_for_variable"),
        "wrong_reverse_param": _error_bucket_summary(eval_df, "wrong_reverse_param"),
        "copy_input_number": _error_bucket_summary(eval_df, "copy_input_number"),
        "by_type": by_type.to_dict("records") if not by_type.empty else [],
        "verifier_summary": verifier_summary,
        "verifier_by_type": verifier_by_type.to_dict("records") if not verifier_by_type.empty else [],
        "error_buckets_by_type": error_by_type.to_dict("records") if not error_by_type.empty else [],
        "train_metrics": CURRICULUM_STAGE_TRAIN_METRICS.get(stage_name, {}),
        "selection_report": CURRICULUM_STAGE_SELECTION_REPORTS.get(stage_name, {}),
        "train_stats": CURRICULUM_STAGE_TRAIN_STATS.get(stage_name, {}),
        "eval_rows_path": str(CURRICULUM_STAGE_EVAL_ROWS_PATHS[stage_name]),
        "eval_rows_json_path": str(CURRICULUM_STAGE_EVAL_ROWS_JSON_PATHS[stage_name]),
        "valid_output_path": str(CURRICULUM_STAGE_VALID_OUTPUT_PATHS[stage_name]),
        "wrong_cases_preview": [r for r in eval_rows if r["score"] == 0][:30],
    }

    save_records_jsonl(eval_rows, CURRICULUM_STAGE_EVAL_ROWS_PATHS[stage_name])
    save_json(eval_rows, CURRICULUM_STAGE_EVAL_ROWS_JSON_PATHS[stage_name])
    save_json(report, CURRICULUM_STAGE_VALID_REPORT_PATHS[stage_name])
    return report, eval_rows


def _load_baseline_reference_score_10():
    if BASELINE_REFERENCE_SCORE_10 is not None:
        return float(BASELINE_REFERENCE_SCORE_10)
    for path in BASELINE_REFERENCE_REPORT_CANDIDATES:
        path = Path(path)
        if not path.exists():
            continue
        try:
            obj = json.loads(path.read_text(encoding="utf-8"))
            if "summary" in obj and "score_10" in obj["summary"]:
                return float(obj["summary"]["score_10"])
            if "overall_score_10" in obj:
                return float(obj["overall_score_10"])
        except Exception as e:
            print("Không đọc được baseline reference:", path, repr(e))
    return None


def _delta(a, b):
    if a is None or b is None:
        return None
    return float(a) - float(b)


def _best_stage(stage_reports, key):
    candidates = [(name, report.get(key)) for name, report in stage_reports.items() if report]
    candidates = [(name, value) for name, value in candidates if value is not None]
    if not candidates:
        return None
    return max(candidates, key=lambda x: x[1])[0]


def _type_regressions(prev_report, curr_report):
    prev = prev_report.get("score_10_by_type", {}) if prev_report else {}
    curr = curr_report.get("score_10_by_type", {}) if curr_report else {}
    regressions = {}
    for rec_type in sorted(set(prev) & set(curr)):
        delta = float(curr[rec_type]) - float(prev[rec_type])
        if delta < -1e-12:
            regressions[rec_type] = delta
    return regressions


CURRICULUM_STAGE_REPORTS = {}
CURRICULUM_STAGE_EVAL_ROWS = {}

if CURRICULUM_STAGE_VALID_OUTPUTS:
    for stage_spec in CURRICULUM_STAGE_SPECS:
        stage_name = stage_spec["name"]
        print("\n" + "=" * 90)
        print(f"Evaluate curriculum stage: {stage_name}")
        stage_report, stage_rows = build_stage_eval_report(stage_name, CURRICULUM_STAGE_VALID_OUTPUTS.get(stage_name, []))
        CURRICULUM_STAGE_REPORTS[stage_name] = stage_report
        CURRICULUM_STAGE_EVAL_ROWS[stage_name] = stage_rows
        print(json.dumps({
            "stage_name": stage_name,
            "overall_score_10": stage_report["overall_score_10"],
            "GSM_all_score": stage_report["GSM_all_score"],
            "MATH_all_score": stage_report["MATH_all_score"],
        }, ensure_ascii=False, indent=2))

    stage1_GSM_direct_report = CURRICULUM_STAGE_REPORTS.get("stage1_GSM_direct")
    stage2_GSM_all_report = CURRICULUM_STAGE_REPORTS.get("stage2_GSM_all")
    stage3_GSM_MATH_direct_report = CURRICULUM_STAGE_REPORTS.get("stage3_GSM_MATH_direct")
    stage4_all_report = CURRICULUM_STAGE_REPORTS.get("stage4_all")

    baseline_reference_score_10 = _load_baseline_reference_score_10()
    transition_regressions = {
        "stage2_vs_stage1": _type_regressions(stage1_GSM_direct_report, stage2_GSM_all_report),
        "stage3_vs_stage2": _type_regressions(stage2_GSM_all_report, stage3_GSM_MATH_direct_report),
        "stage4_vs_stage3": _type_regressions(stage3_GSM_MATH_direct_report, stage4_all_report),
    }

    GSM_score_after_stage1 = stage1_GSM_direct_report.get("GSM_all_score") if stage1_GSM_direct_report else None
    GSM_score_after_stage2 = stage2_GSM_all_report.get("GSM_all_score") if stage2_GSM_all_report else None
    GSM_score_after_stage3 = stage3_GSM_MATH_direct_report.get("GSM_all_score") if stage3_GSM_MATH_direct_report else None
    GSM_score_after_stage4 = stage4_all_report.get("GSM_all_score") if stage4_all_report else None

    curriculum_summary = {
        "GSM_score_after_stage1": GSM_score_after_stage1,
        "GSM_score_after_stage2": GSM_score_after_stage2,
        "GSM_score_after_stage3": GSM_score_after_stage3,
        "GSM_score_after_stage4": GSM_score_after_stage4,
        "forgetting_GSM_stage3": _delta(GSM_score_after_stage1, GSM_score_after_stage3),
        "forgetting_GSM_stage4": _delta(GSM_score_after_stage1, GSM_score_after_stage4),
        "type_regression_count": int(sum(len(v) for v in transition_regressions.values())),
        "type_regressions_by_transition": transition_regressions,
        "delta_stage1_vs_baseline": _delta(stage1_GSM_direct_report.get("overall_score_10") if stage1_GSM_direct_report else None, baseline_reference_score_10),
        "delta_stage2_vs_stage1": _delta(stage2_GSM_all_report.get("overall_score_10") if stage2_GSM_all_report else None, stage1_GSM_direct_report.get("overall_score_10") if stage1_GSM_direct_report else None),
        "delta_stage3_vs_stage2": _delta(stage3_GSM_MATH_direct_report.get("overall_score_10") if stage3_GSM_MATH_direct_report else None, stage2_GSM_all_report.get("overall_score_10") if stage2_GSM_all_report else None),
        "delta_stage4_vs_stage3": _delta(stage4_all_report.get("overall_score_10") if stage4_all_report else None, stage3_GSM_MATH_direct_report.get("overall_score_10") if stage3_GSM_MATH_direct_report else None),
        "best_stage_by_overall": _best_stage(CURRICULUM_STAGE_REPORTS, "overall_score_10"),
        "best_stage_by_GSM": _best_stage(CURRICULUM_STAGE_REPORTS, "GSM_all_score"),
        "best_stage_by_MATH": _best_stage(CURRICULUM_STAGE_REPORTS, "MATH_all_score"),
        "train_samples_by_type_per_stage": {name: stats["train_samples_by_type"] for name, stats in CURRICULUM_STAGE_TRAIN_STATS.items()},
        "train_tokens_by_type_per_stage": {name: stats["train_tokens_by_type"] for name, stats in CURRICULUM_STAGE_TRAIN_STATS.items()},
        "avg_length_by_type_per_stage": {name: stats["avg_length_by_type"] for name, stats in CURRICULUM_STAGE_TRAIN_STATS.items()},
        "effective_sampling_ratio_by_type": {name: stats["effective_sampling_ratio_by_type"] for name, stats in CURRICULUM_STAGE_TRAIN_STATS.items()},
        "steps_per_type": {name: stats["steps_per_type"] for name, stats in CURRICULUM_STAGE_TRAIN_STATS.items()},
        "baseline_reference_score_10": baseline_reference_score_10,
    }

    final_stage_report = stage4_all_report or {}
    eval_rows = CURRICULUM_STAGE_EVAL_ROWS.get("stage4_all", [])
    summary = final_stage_report.get("summary")
    save_records_jsonl(eval_rows, VALID_EVAL_ROWS_PATH)
    save_json(eval_rows, VALID_EVAL_ROWS_JSON_PATH)

    report = {
        "variant_name": VARIANT_NAME,
        "experiment_name": EXPERIMENT_NAME,
        "curriculum_summary": curriculum_summary,
        "stage1_GSM_direct_report": stage1_GSM_direct_report,
        "stage2_GSM_all_report": stage2_GSM_all_report,
        "stage3_GSM_MATH_direct_report": stage3_GSM_MATH_direct_report,
        "stage4_all_report": stage4_all_report,
        "stage_reports": CURRICULUM_STAGE_REPORTS,
        "config": {
            "epochs_fallback": EPOCHS,
            "stage_epochs": CURRICULUM_STAGE_EPOCHS,
            "stage_max_steps": CURRICULUM_STAGE_MAX_STEPS,
            "stage_max_samples": CURRICULUM_STAGE_MAX_SAMPLES,
            "curriculum_stage_specs": CURRICULUM_STAGE_SPECS,
            "sample_with_replacement": CURRICULUM_SAMPLE_WITH_REPLACEMENT,
            "experiment_name": EXPERIMENT_NAME,
            "target_mode": TARGET_MODE,
            "answer_only_ratio": ANSWER_ONLY_RATIO,
            "instruction_by_target_mode": INSTRUCTION_BY_TARGET_MODE,
            "fast_train_subset": FAST_TRAIN_SUBSET,
            "max_train_records_for_speed": MAX_TRAIN_RECORDS_FOR_SPEED,
            "keep_all_types_for_speed": sorted(KEEP_ALL_TYPES_FOR_SPEED),
            "type_quotas_for_speed": TYPE_QUOTAS_FOR_SPEED,
            "fast_subset_strategy": FAST_SUBSET_STRATEGY,
            "speed_subset_report": speed_subset_report,
            "eval_during_train": EVAL_DURING_TRAIN,
            "eval_loss_after_each_stage": EVAL_LOSS_AFTER_EACH_STAGE,
            "save_during_train": SAVE_DURING_TRAIN,
            "lr": LEARNING_RATE,
            "effective_batch": effective_batch,
            "max_length": MAX_LENGTH,
            "max_new_tokens": MAX_NEW_TOKENS,
            "num_beams": NUM_BEAMS,
            "do_sample": DO_SAMPLE,
            "source": BASELINE_SOURCE if "BASELINE_SOURCE" in globals() else VARIANT_NAME,
            "decoding": BASELINE_DECODING if "BASELINE_DECODING" in globals() else None,
            "prompt_template": PROMPT_TEMPLATE,
            "instruction": INSTRUCTION,
            "task_group_map": TASK_GROUP_MAP,
            "type_label_map": TYPE_LABEL_MAP,
            "use_lora": USE_LORA,
            "lora_r": LORA_R if USE_LORA else None,
            "lora_alpha": LORA_ALPHA if USE_LORA else None,
            "lora_dropout": LORA_DROPOUT if USE_LORA else None,
            "lora_target_modules": LORA_TARGET_MODULES if USE_LORA else None,
            "run_mode": "variant_4_curriculum_gsm_to_math_no_stepfmt_no_self_consistency_no_rule_verifier",
        },
        "final_stage_eval_rows_path": str(VALID_EVAL_ROWS_PATH),
        "final_stage_eval_rows_json_path": str(VALID_EVAL_ROWS_JSON_PATH),
    }
    save_json(report, VALID_REPORT_PATH)
    print(f"\nWrote curriculum report: {VALID_REPORT_PATH}")
    print(json.dumps(curriculum_summary, ensure_ascii=False, indent=2))

    if eval_rows:
        show_cases("Stage 4: Một vài case đúng (score=10)", [r for r in eval_rows if r["score"] == 10])
        show_cases("Stage 4: Một vài case sai (score=0)", [r for r in eval_rows if r["score"] == 0 and r["extractable"]])
else:
    eval_rows = []
    summary = None
    stage1_GSM_direct_report = None
    stage2_GSM_all_report = None
    stage3_GSM_MATH_direct_report = None
    stage4_all_report = None
    print("Không có validation output theo stage để đánh giá")


In [ ]:
# 14. Sinh test_predictions.json cho Phase 2 từ checkpoint stage4_all
RUN_TEST_INFERENCE = True
MODEL_FOR_INFERENCE = CURRICULUM_STAGE_OUTPUT_DIRS.get("stage4_all", OUTPUT_DIR)

if RUN_TEST_INFERENCE and test_records:
    if not Path(MODEL_FOR_INFERENCE).exists():
        print("Checkpoint stage4_all chưa tồn tại, bỏ qua test inference:", MODEL_FOR_INFERENCE)
    else:
        test_outputs = generate_predictions(MODEL_FOR_INFERENCE, test_records, TEST_OUTPUT_PATH, name="test:stage4_all")
        test_outputs_clean = [{k: v for k, v in o.items() if not k.startswith("_")} for o in test_outputs]
        for row in test_outputs_clean:
            row["stage_name"] = "stage4_all"
        save_json(test_outputs_clean, TEST_OUTPUT_PATH)
        print("\nTest output mẫu:")
        print(json.dumps(test_outputs_clean[:2], ensure_ascii=False, indent=2)[:1200])
else:
    print("Không có test.json, bỏ qua bước test inference")


In [ ]:
# 15. Kiểm tra file đầu ra Variant 4
paths_to_check = [
    WORK_DIR,
    STAGE_OUTPUT_ROOT,
    STAGE_PREDICTION_DIR,
    STAGE_REPORT_DIR,
    OUTPUT_DIR,
    VALID_OUTPUT_PATH,
    VALID_EVAL_ROWS_PATH,
    VALID_REPORT_PATH,
    TEST_OUTPUT_PATH,
]

for stage_name in [s["name"] for s in CURRICULUM_STAGE_SPECS]:
    paths_to_check.extend([
        CURRICULUM_STAGE_OUTPUT_DIRS[stage_name],
        CURRICULUM_STAGE_VALID_OUTPUT_PATHS[stage_name],
        CURRICULUM_STAGE_VALID_REPORT_PATHS[stage_name],
        CURRICULUM_STAGE_EVAL_ROWS_PATHS[stage_name],
    ])

seen = set()
for p in paths_to_check:
    p = Path(p)
    key = str(p)
    if key in seen:
        continue
    seen.add(key)
    if p.exists():
        size = p.stat().st_size if p.is_file() else "<dir>"
        print(p, "|", size)

print("\nDone Variant 4 curriculum GSM -> MATH.")
